In [59]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

In [60]:
from pathlib import Path

base_dir = Path.cwd()

raw_dir = base_dir / "Raw_Data"
diag_dir = base_dir / "Data_Cleaned" / "Diagnostics"

diag_dir.mkdir(parents=True, exist_ok=True)

print("Base directory:", base_dir)
print("Raw folder exists:", raw_dir.exists())
print("Diagnostics folder:", diag_dir)

Base directory: /home/sagemaker-user
Raw folder exists: True
Diagnostics folder: /home/sagemaker-user/Data_Cleaned/Diagnostics


In [61]:
files = [
    "A_Interval.csv",
    "A-Daily.csv",
    "B_Interval.csv",
    "B-Daily.csv",
    "C_Interval.csv",
    "C-Daily.csv",
    "D_Interval.csv",
    "D-Daily.csv",
    "Daily_Staffing.csv"
]

pd.DataFrame({
    "file_name": files,
    "exists": [(raw_dir / f).exists() for f in files]
})

,file_name,exists
0,A_Interval.csv,True
1,A-Daily.csv,True
2,B_Interval.csv,True
3,B-Daily.csv,True
4,C_Interval.csv,True
5,C-Daily.csv,True
6,D_Interval.csv,True
7,D-Daily.csv,True
8,Daily_Staffing.csv,True


In [62]:
def clean_col_name(col):
    col = str(col).strip().lower()
    col = re.sub(r"[^\w\s]", "", col)   # remove punctuation
    col = re.sub(r"\s+", "_", col)      # spaces to underscore
    col = re.sub(r"_+", "_", col)       # collapse multiple underscores
    return col.strip("_")

In [63]:
def standardize_columns(df):
    df = df.copy()
    new_cols = [clean_col_name(c) for c in df.columns]

    seen = {}
    final_cols = []

    for col in new_cols:
        if col not in seen:
            seen[col] = 0
            final_cols.append(col)
        else:
            seen[col] += 1
            final_cols.append(f"{col}_{seen[col]}")

    df.columns = final_cols
    return df

In [64]:
def replace_null_like_values(df):
    df = df.copy()

    null_tokens = ["", " ", "na", "n/a", "null", "none", "-", "?"]

    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).str.strip()
            df[col] = df[col].replace(null_tokens, np.nan)

    return df

In [65]:
def get_dataset_type(file_name):
    name = file_name.lower()
    if "interval" in name:
        return "interval"
    elif "staffing" in name:
        return "staffing"
    elif "daily" in name:
        return "daily"
    return "unknown"

In [66]:
def get_portfolio(file_name):
    match = re.match(r"([A-D])[_-]", file_name)
    if match:
        return match.group(1)
    return "ALL"

In [67]:
# Loading all the data
data = {}

for file_name in files:
    file_path = raw_dir / file_name

    try:
        df = pd.read_csv(file_path)
        df = standardize_columns(df)
        df = replace_null_like_values(df)
        data[file_name] = df

        print(f"{file_name} loaded successfully | shape = {df.shape}")
    except Exception as e:
        print(f"{file_name} failed to load")
        print("Reason:", e)

A_Interval.csv loaded successfully | shape = (4084, 8)
A-Daily.csv loaded successfully | shape = (731, 5)
B_Interval.csv loaded successfully | shape = (4293, 8)
B-Daily.csv loaded successfully | shape = (731, 5)
C_Interval.csv loaded successfully | shape = (4368, 8)
C-Daily.csv loaded successfully | shape = (731, 5)
D_Interval.csv loaded successfully | shape = (4358, 8)
D-Daily.csv loaded successfully | shape = (731, 5)
Daily_Staffing.csv loaded successfully | shape = (365, 5)


In [68]:
for file_name, df in data.items():
    print("\n" + "=" * 90)
    print(file_name)
    print("=" * 90)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(3))


A_Interval.csv
Shape: (4084, 8)
Columns: ['month', 'day', 'interval', 'service_level', 'call_volume', 'abandoned_calls', 'abandoned_rate', 'cct']


,month,day,interval,service_level,call_volume,abandoned_calls,abandoned_rate,cct
0,April,1.0,0:00,100.00%,5.0,0.0,0.00%,137.6
1,April,1.0,0:30,100.00%,5.0,0.0,0.00%,263.4
2,April,1.0,1:00,100.00%,4.0,0.0,0.00%,333.25



A-Daily.csv
Shape: (731, 5)
Columns: ['date', 'call_volume', 'cct', 'service_level', 'abandon_rate']


,date,call_volume,cct,service_level,abandon_rate
0,01/01/24 Mon,"2,147",302.45,98.55%,0.37%
1,01/02/24 Tue,"7,458",349.22,52.13%,11.36%
2,01/03/24 Wed,"6,882",331.07,70.46%,4.32%



B_Interval.csv
Shape: (4293, 8)
Columns: ['month', 'day', 'interval', 'service_level', 'call_volume', 'abandoned_calls', 'abandoned_rate', 'cct']


...

,month,day,interval,service_level,call_volume,abandoned_calls,abandoned_rate,cct
0,April,1,0:00,100.00%,30.0,0.0,0.00%,306
1,April,1,0:30,100.00%,29.0,0.0,0.00%,353.14
2,April,1,1:00,100.00%,25.0,0.0,0.00%,373.76



D-Daily.csv
Shape: (731, 5)
Columns: ['date', 'call_volume', 'cct', 'service_level', 'abandon_rate']


,date,call_volume,cct,service_level,abandon_rate
0,01/01/24 Mon,"4,380",333.91,99.38%,0.32%
1,01/02/24 Tue,"16,184",348.04,55.53%,7.10%
2,01/03/24 Wed,"15,575",333.97,59.45%,6.79%



Daily_Staffing.csv
Shape: (365, 5)
Columns: ['unnamed_0', 'a', 'b', 'c', 'd']


,unnamed_0,a,b,c,d
0,1/1/25,47.0,75.0,353.0,143.0
1,1/2/25,82.0,184.0,491.0,195.0
2,1/3/25,92.0,186.0,462.0,183.0


In [69]:
# Summary
file_summary = []

for file_name, df in data.items():
    n_rows, n_cols = df.shape
    total_cells = n_rows * n_cols
    total_missing = int(df.isna().sum().sum())
    duplicate_rows = int(df.duplicated().sum())

    file_summary.append({
        "filename": file_name,
        "dataset_type": get_dataset_type(file_name),
        "portfolio": get_portfolio(file_name),
        "rows": n_rows,
        "columns": n_cols,
        "duplicate_rows": duplicate_rows,
        "total_missing_cells": total_missing,
        "total_missing_pct": round((total_missing / total_cells) * 100, 2) if total_cells else 0
    })

file_summary_df = pd.DataFrame(file_summary)
display(file_summary_df)

,filename,dataset_type,portfolio,rows,columns,duplicate_rows,total_missing_cells,total_missing_pct
0,A_Interval.csv,interval,A,4084,8,6,197,0.60
1,A-Daily.csv,daily,A,731,5,0,24,0.66
2,B_Interval.csv,interval,B,4293,8,6,217,0.63
3,B-Daily.csv,daily,B,731,5,0,29,0.79
4,C_Interval.csv,interval,C,4368,8,0,182,0.52
5,C-Daily.csv,daily,C,731,5,0,14,0.38
6,D_Interval.csv,interval,D,4358,8,0,152,0.44
7,D-Daily.csv,daily,D,731,5,0,32,0.88
8,Daily_Staffing.csv,staffing,ALL,365,5,0,135,7.40


In [70]:
column_summary = []

for file_name, df in data.items():
    for col in df.columns:
        column_summary.append({
            "filename": file_name,
            "dataset_type": get_dataset_type(file_name),
            "portfolio": get_portfolio(file_name),
            "column_name": col,
            "dtype": str(df[col].dtype),
            "missing_count": int(df[col].isna().sum()),
            "missing_pct": round(df[col].isna().mean() * 100, 2),
            "unique_non_null": int(df[col].nunique(dropna=True))
        })

column_summary_df = pd.DataFrame(column_summary)
display(column_summary_df.head(20))

,filename,dataset_type,portfolio,column_name,dtype,missing_count,missing_pct,unique_non_null
0,A_Interval.csv,interval,A,month,object,0,0.00,4
1,A_Interval.csv,interval,A,day,float64,1,0.02,31
2,A_Interval.csv,interval,A,interval,object,0,0.00,49
3,A_Interval.csv,interval,A,service_level,object,0,0.00,1017
4,A_Interval.csv,interval,A,call_volume,float64,89,2.18,296
5,A_Interval.csv,interval,A,abandoned_calls,float64,107,2.62,21
6,A_Interval.csv,interval,A,abandoned_rate,object,0,0.00,382
7,A_Interval.csv,interval,A,cct,object,0,0.00,3298
8,A-Daily.csv,daily,A,date,object,0,0.00,731
9,A-Daily.csv,daily,A,call_volume,object,0,0.00,643


In [71]:
worst_missing_cols = column_summary_df.sort_values(
    by=["missing_pct", "filename"],
    ascending=[False, True]
)

display(worst_missing_cols.head(30))

,filename,dataset_type,portfolio,column_name,dtype,missing_count,missing_pct,unique_non_null
55,Daily_Staffing.csv,staffing,ALL,c,float64,37,10.14,196
56,Daily_Staffing.csv,staffing,ALL,d,float64,36,9.86,114
54,Daily_Staffing.csv,staffing,ALL,b,float64,35,9.59,125
53,Daily_Staffing.csv,staffing,ALL,a,float64,27,7.40,71
49,D-Daily.csv,daily,D,cct,float64,32,4.38,653
23,B-Daily.csv,daily,B,cct,float64,29,3.97,664
10,A-Daily.csv,daily,A,cct,float64,24,3.28,665
5,A_Interval.csv,interval,A,abandoned_calls,float64,107,2.62,21
18,B_Interval.csv,interval,B,abandoned_calls,float64,109,2.54,53
17,B_Interval.csv,interval,B,call_volume,float64,108,2.52,516


In [72]:
date_summary = []

for file_name, df in data.items():
    dataset_type = get_dataset_type(file_name)

    if dataset_type in ["daily", "staffing"]:
        if "date" in df.columns:
            dt = pd.to_datetime(df["date"], errors="coerce")

            date_summary.append({
                "filename": file_name,
                "dataset_type": dataset_type,
                "portfolio": get_portfolio(file_name),
                "min_date": dt.min(),
                "max_date": dt.max(),
                "invalid_date_count": int(dt.isna().sum()),
                "duplicate_date_count": int(dt.duplicated().sum())
            })
        else:
            date_summary.append({
                "filename": file_name,
                "dataset_type": dataset_type,
                "portfolio": get_portfolio(file_name),
                "min_date": None,
                "max_date": None,
                "invalid_date_count": None,
                "duplicate_date_count": None
            })

date_summary_df = pd.DataFrame(date_summary)
display(date_summary_df)

/tmp/ipykernel_14136/1474032895.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(df["date"], errors="coerce")
/tmp/ipykernel_14136/1474032895.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(df["date"], errors="coerce")
/tmp/ipykernel_14136/1474032895.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(df["date"], errors="coerce")
/tmp/ipykernel_14136/1474032895.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent a

,filename,dataset_type,portfolio,min_date,max_date,invalid_date_count,duplicate_date_count
0,A-Daily.csv,daily,A,2024-01-01,2025-12-31,0.0,0.0
1,B-Daily.csv,daily,B,2024-01-01,2025-12-31,0.0,0.0
2,C-Daily.csv,daily,C,2024-01-01,2025-12-31,0.0,0.0
3,D-Daily.csv,daily,D,2024-01-01,2025-12-31,0.0,0.0
4,Daily_Staffing.csv,staffing,ALL,NaT,NaT,NaN,NaN


In [73]:
for file_name, df in data.items():
    print(file_name, "->", get_dataset_type(file_name), "| shape =", df.shape)

A_Interval.csv -> interval | shape = (4084, 8)
A-Daily.csv -> daily | shape = (731, 5)
B_Interval.csv -> interval | shape = (4293, 8)
B-Daily.csv -> daily | shape = (731, 5)
C_Interval.csv -> interval | shape = (4368, 8)
C-Daily.csv -> daily | shape = (731, 5)
D_Interval.csv -> interval | shape = (4358, 8)
D-Daily.csv -> daily | shape = (731, 5)
Daily_Staffing.csv -> staffing | shape = (365, 5)


In [74]:
interval_completeness_list = []

In [75]:
interval_completeness_list = []

for file_name, df in data.items():
    if get_dataset_type(file_name) == "interval":
        print(f"\nChecking {file_name}...")

        required_cols = {"month", "day", "interval"}

        print("Columns:", df.columns.tolist())

        if required_cols.issubset(df.columns):
            temp = df.copy()

            temp["month_clean"] = temp["month"].astype(str).str.strip()
            temp["day_num"] = pd.to_numeric(temp["day"], errors="coerce")
            temp["interval_num"] = pd.to_numeric(temp["interval"], errors="coerce")

            print("Non-null month values:", temp["month_clean"].notna().sum())
            print("Non-null day values:", temp["day_num"].notna().sum())
            print("Non-null interval values:", temp["interval_num"].notna().sum())

            day_counts = (
                temp.groupby(["month_clean", "day_num"], dropna=False)["interval_num"]
                .nunique()
                .reset_index(name="unique_intervals")
                .sort_values(["month_clean", "day_num"])
            )

            day_counts["filename"] = file_name
            day_counts["portfolio"] = get_portfolio(file_name)
            day_counts["is_incomplete_day"] = day_counts["unique_intervals"] < 48

            interval_completeness_list.append(day_counts)

            print("Rows added:", len(day_counts))
        else:
            print("Missing required columns:", required_cols - set(df.columns))


Checking A_Interval.csv...
Columns: ['month', 'day', 'interval', 'service_level', 'call_volume', 'abandoned_calls', 'abandoned_rate', 'cct']
Non-null month values: 4084
Non-null day values: 4083
Non-null interval values: 0
Rows added: 92

Checking B_Interval.csv...
Columns: ['month', 'day', 'interval', 'service_level', 'call_volume', 'abandoned_calls', 'abandoned_rate', 'cct']
Non-null month values: 4293
Non-null day values: 4293
Non-null interval values: 0
Rows added: 91

Checking C_Interval.csv...
Columns: ['month', 'day', 'interval', 'service_level', 'call_volume', 'abandoned_calls', 'abandoned_rate', 'cct']
Non-null month values: 4368
Non-null day values: 4368
Non-null interval values: 0
Rows added: 91

Checking D_Interval.csv...
Columns: ['month', 'day', 'interval', 'service_level', 'call_volume', 'abandoned_calls', 'abandoned_rate', 'cct']
Non-null month values: 4358
Non-null day values: 4358
Non-null interval values: 0
Rows added: 91


In [76]:
print("Number of dfs inside interval_completeness_list:", len(interval_completeness_list))

Number of dfs inside interval_completeness_list: 4


In [77]:
interval_completeness_df = pd.concat(interval_completeness_list, ignore_index=True)

print("Shape:", interval_completeness_df.shape)
display(interval_completeness_df.head(20))

Shape: (365, 6)


,month_clean,day_num,unique_intervals,filename,portfolio,is_incomplete_day
0,April,1.0,0,A_Interval.csv,A,True
1,April,2.0,0,A_Interval.csv,A,True
2,April,3.0,0,A_Interval.csv,A,True
3,April,4.0,0,A_Interval.csv,A,True
4,April,5.0,0,A_Interval.csv,A,True
5,April,6.0,0,A_Interval.csv,A,True
6,April,7.0,0,A_Interval.csv,A,True
7,April,8.0,0,A_Interval.csv,A,True
8,April,9.0,0,A_Interval.csv,A,True
9,April,10.0,0,A_Interval.csv,A,True


In [78]:
incomplete_day_counts = (
    interval_completeness_df
    .groupby(["portfolio", "filename"], as_index=False)["is_incomplete_day"]
    .sum()
    .rename(columns={"is_incomplete_day": "incomplete_days"})
)

display(incomplete_day_counts)

,portfolio,filename,incomplete_days
0,A,A_Interval.csv,92
1,B,B_Interval.csv,91
2,C,C_Interval.csv,91
3,D,D_Interval.csv,91


In [79]:
missing_by_file = (
    column_summary_df
    .groupby(["filename", "dataset_type", "portfolio"], as_index=False)
    .agg(
        total_columns=("column_name", "count"),
        columns_with_missing=("missing_count", lambda x: (x > 0).sum()),
        max_missing_pct=("missing_pct", "max"),
        avg_missing_pct=("missing_pct", "mean")
    )
    .sort_values(["dataset_type", "portfolio", "filename"])
)

display(missing_by_file)

,filename,dataset_type,portfolio,total_columns,columns_with_missing,max_missing_pct,avg_missing_pct
0,A-Daily.csv,daily,A,5,1,3.28,0.65600
2,B-Daily.csv,daily,B,5,1,3.97,0.79400
4,C-Daily.csv,daily,C,5,1,1.92,0.38400
6,D-Daily.csv,daily,D,5,1,4.38,0.87600
1,A_Interval.csv,interval,A,8,3,2.62,0.60250
3,B_Interval.csv,interval,B,8,2,2.54,0.63250
5,C_Interval.csv,interval,C,8,3,1.79,0.52125
7,D_Interval.csv,interval,D,8,2,1.81,0.43625
8,Daily_Staffing.csv,staffing,ALL,5,4,10.14,7.39800


In [80]:
top_missing_per_file = (
    column_summary_df[column_summary_df["missing_count"] > 0]
    .sort_values(["filename", "missing_pct"], ascending=[True, False])
    .groupby("filename")
    .head(5)
    .reset_index(drop=True)
)

display(top_missing_per_file)

,filename,dataset_type,portfolio,column_name,dtype,missing_count,missing_pct,unique_non_null
0,A-Daily.csv,daily,A,cct,float64,24,3.28,665
1,A_Interval.csv,interval,A,abandoned_calls,float64,107,2.62,21
2,A_Interval.csv,interval,A,call_volume,float64,89,2.18,296
3,A_Interval.csv,interval,A,day,float64,1,0.02,31
4,B-Daily.csv,daily,B,cct,float64,29,3.97,664
5,B_Interval.csv,interval,B,abandoned_calls,float64,109,2.54,53
6,B_Interval.csv,interval,B,call_volume,float64,108,2.52,516
7,C-Daily.csv,daily,C,cct,float64,14,1.92,664
8,C_Interval.csv,interval,C,call_volume,float64,78,1.79,1041
9,C_Interval.csv,interval,C,cct,float64,55,1.26,3677


In [81]:
interval_key_check = []

for file_name, df in data.items():
    if get_dataset_type(file_name) == "interval":
        required_cols = {"month", "day", "interval"}

        if required_cols.issubset(df.columns):
            temp = df.copy()

            temp["month_clean"] = temp["month"].astype(str).str.strip()
            temp["day_num"] = pd.to_numeric(temp["day"], errors="coerce")
            temp["interval_num"] = pd.to_numeric(temp["interval"], errors="coerce")

            dup_count = temp.duplicated(
                subset=["month_clean", "day_num", "interval_num"]
            ).sum()

            interval_key_check.append({
                "filename": file_name,
                "portfolio": get_portfolio(file_name),
                "duplicate_month_day_interval_rows": int(dup_count)
            })

interval_key_check_df = pd.DataFrame(interval_key_check)
display(interval_key_check_df)

,filename,portfolio,duplicate_month_day_interval_rows
0,A_Interval.csv,A,3992
1,B_Interval.csv,B,4202
2,C_Interval.csv,C,4277
3,D_Interval.csv,D,4267


In [82]:
print("File summary shape:", file_summary_df.shape)
print("Column summary shape:", column_summary_df.shape)
print("Date summary shape:", date_summary_df.shape)
print("Interval completeness shape:", interval_completeness_df.shape)
print("Incomplete day counts shape:", incomplete_day_counts.shape)
print("Interval key check shape:", interval_key_check_df.shape)

File summary shape: (9, 8)
Column summary shape: (57, 8)
Date summary shape: (5, 7)
Interval completeness shape: (365, 6)
Incomplete day counts shape: (4, 3)
Interval key check shape: (4, 3)


In [83]:
file_summary_df.to_csv(diag_dir / "phase1_file_summary.csv", index=False)
column_summary_df.to_csv(diag_dir / "phase1_column_summary.csv", index=False)
date_summary_df.to_csv(diag_dir / "phase1_date_summary.csv", index=False)
interval_completeness_df.to_csv(diag_dir / "phase1_interval_completeness.csv", index=False)
incomplete_day_counts.to_csv(diag_dir / "phase1_incomplete_day_counts.csv", index=False)
interval_key_check_df.to_csv(diag_dir / "phase1_interval_key_check.csv", index=False)
missing_by_file.to_csv(diag_dir / "phase1_missing_by_file.csv", index=False)
top_missing_per_file.to_csv(diag_dir / "phase1_top_missing_per_file.csv", index=False)

print("Phase 1 diagnostics saved successfully in:")
print(diag_dir)
print(sorted([p.name for p in diag_dir.iterdir()]))

Phase 1 diagnostics saved successfully in:
/home/sagemaker-user/Data_Cleaned/Diagnostics
['.ipynb_checkpoints', 'phase1_column_summary.csv', 'phase1_date_summary.csv', 'phase1_file_summary.csv', 'phase1_incomplete_day_counts.csv', 'phase1_interval_completeness.csv', 'phase1_interval_key_check.csv', 'phase1_missing_by_file.csv', 'phase1_observations.csv', 'phase1_top_missing_per_file.csv']


In [84]:
phase2_dir = base_dir / "Data_Cleaned" / "Phase2_Standardized"
phase2_dir.mkdir(parents=True, exist_ok=True)
print("Phase 2 output directory:", phase2_dir)

Phase 2 output directory: /home/sagemaker-user/Data_Cleaned/Phase2_Standardized


In [86]:
def standardize_columns(df):
    df = df.copy()
    cleaned_cols = [clean_col_name(c) for c in df.columns]

    seen = {}
    final_cols = []

    for col in cleaned_cols:
        if col not in seen:
            seen[col] = 0
            final_cols.append(col)
        else:
            seen[col] += 1
            final_cols.append(f"{col}_{seen[col]}")

    df.columns = final_cols
    return df

In [87]:
def standardize_object_values(df):
    df = df.copy()

    null_tokens = {"", " ", "na", "n/a", "null", "none", "-", "?", "nan"}

    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).str.strip()
            df[col] = df[col].replace(list(null_tokens), np.nan)

    return df

In [88]:
def get_dataset_type(file_name):
    name = file_name.lower()
    if "interval" in name:
        return "interval"
    elif "staffing" in name:
        return "staffing"
    elif "daily" in name:
        return "daily"
    return "unknown"

In [89]:
numeric_columns_by_type = {
    "daily": ["call_volume", "cct", "service_level", "abandon_rate"],
    "interval": ["month", "day", "interval", "service_level", "call_volume", "abandoned_calls", "abandoned_rate", "cct"],
    "staffing": []
}

In [90]:
def convert_numeric_columns(df, dataset_type):
    df = df.copy()

    cols_to_convert = numeric_columns_by_type.get(dataset_type, []).copy()

    if dataset_type == "staffing":
        cols_to_convert = [c for c in df.columns if c != "date"]

    for col in cols_to_convert:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

In [91]:
phase2_summary = []
standardized_data = {}

for raw_name, clean_name in file_rename_map.items():
    raw_path = raw_dir / raw_name

    if not raw_path.exists():
        print(f"Skipping {raw_name} because file was not found.")
        continue

    df = pd.read_csv(raw_path)

    # standardize column names
    df = standardize_columns(df)

    # standardize string values and null-like tokens
    df = standardize_object_values(df)

    # detect dataset type
    dataset_type = get_dataset_type(clean_name)

    # convert numeric columns carefully
    df = convert_numeric_columns(df, dataset_type)

    # save cleaned standardized copy
    save_path = phase2_dir / clean_name
    df.to_csv(save_path, index=False)

    standardized_data[clean_name] = df.copy()

    phase2_summary.append({
        "raw_name": raw_name,
        "saved_name": clean_name,
        "dataset_type": dataset_type,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "saved_path": str(save_path)
    })

    print(f"Saved {clean_name} | shape = {df.shape}")

Saved A_Daily.csv | shape = (731, 5)
Saved A_Interval.csv | shape = (4084, 8)
Saved B_Daily.csv | shape = (731, 5)
Saved B_Interval.csv | shape = (4293, 8)
Saved C_Daily.csv | shape = (731, 5)
Saved C_Interval.csv | shape = (4368, 8)
Saved D_Daily.csv | shape = (731, 5)
Saved D_Interval.csv | shape = (4358, 8)
Saved Daily_Staffing.csv | shape = (365, 5)


In [92]:
phase2_summary_df = pd.DataFrame(phase2_summary)
display(phase2_summary_df)

,raw_name,saved_name,dataset_type,rows,columns,saved_path
0,A-Daily.csv,A_Daily.csv,daily,731,5,/home/sagemaker-user/Data_Cleaned/Phase2_Stand...
1,A_Interval.csv,A_Interval.csv,interval,4084,8,/home/sagemaker-user/Data_Cleaned/Phase2_Stand...
2,B-Daily.csv,B_Daily.csv,daily,731,5,/home/sagemaker-user/Data_Cleaned/Phase2_Stand...
3,B_Interval.csv,B_Interval.csv,interval,4293,8,/home/sagemaker-user/Data_Cleaned/Phase2_Stand...
4,C-Daily.csv,C_Daily.csv,daily,731,5,/home/sagemaker-user/Data_Cleaned/Phase2_Stand...
5,C_Interval.csv,C_Interval.csv,interval,4368,8,/home/sagemaker-user/Data_Cleaned/Phase2_Stand...
6,D-Daily.csv,D_Daily.csv,daily,731,5,/home/sagemaker-user/Data_Cleaned/Phase2_Stand...
7,D_Interval.csv,D_Interval.csv,interval,4358,8,/home/sagemaker-user/Data_Cleaned/Phase2_Stand...
8,Daily_Staffing.csv,Daily_Staffing.csv,staffing,365,5,/home/sagemaker-user/Data_Cleaned/Phase2_Stand...


In [93]:
for file_name, df in standardized_data.items():
    print("\n" + "=" * 90)
    print(file_name)
    print("=" * 90)
    print("Columns:", df.columns.tolist())
    display(df.head(3))


A_Daily.csv
Columns: ['date', 'call_volume', 'cct', 'service_level', 'abandon_rate']


,date,call_volume,cct,service_level,abandon_rate
0,01/01/24 Mon,NaN,302.45,NaN,NaN
1,01/02/24 Tue,NaN,349.22,NaN,NaN
2,01/03/24 Wed,NaN,331.07,NaN,NaN



A_Interval.csv
Columns: ['month', 'day', 'interval', 'service_level', 'call_volume', 'abandoned_calls', 'abandoned_rate', 'cct']


,month,day,interval,service_level,call_volume,abandoned_calls,abandoned_rate,cct
0,NaN,1.0,NaN,NaN,5.0,0.0,NaN,137.60
1,NaN,1.0,NaN,NaN,5.0,0.0,NaN,263.40
2,NaN,1.0,NaN,NaN,4.0,0.0,NaN,333.25



B_Daily.csv
Columns: ['date', 'call_volume', 'cct', 'service_level', 'abandon_rate']


...

,date,call_volume,cct,service_level,abandon_rate
0,01/01/24 Mon,NaN,333.91,NaN,NaN
1,01/02/24 Tue,NaN,348.04,NaN,NaN
2,01/03/24 Wed,NaN,333.97,NaN,NaN



D_Interval.csv
Columns: ['month', 'day', 'interval', 'service_level', 'call_volume', 'abandoned_calls', 'abandoned_rate', 'cct']


,month,day,interval,service_level,call_volume,abandoned_calls,abandoned_rate,cct
0,NaN,1,NaN,NaN,30.0,0.0,NaN,306.00
1,NaN,1,NaN,NaN,29.0,0.0,NaN,353.14
2,NaN,1,NaN,NaN,25.0,0.0,NaN,373.76



Daily_Staffing.csv
Columns: ['unnamed_0', 'a', 'b', 'c', 'd']


,unnamed_0,a,b,c,d
0,NaN,47.0,75.0,353.0,143.0
1,NaN,82.0,184.0,491.0,195.0
2,NaN,92.0,186.0,462.0,183.0


In [94]:
phase2_dtype_summary = []

for file_name, df in standardized_data.items():
    for col in df.columns:
        phase2_dtype_summary.append({
            "filename": file_name,
            "column_name": col,
            "dtype": str(df[col].dtype),
            "missing_count": int(df[col].isna().sum())
        })

phase2_dtype_summary_df = pd.DataFrame(phase2_dtype_summary)
display(phase2_dtype_summary_df.head(30))

,filename,column_name,dtype,missing_count
0,A_Daily.csv,date,object,0
1,A_Daily.csv,call_volume,float64,724
2,A_Daily.csv,cct,float64,24
3,A_Daily.csv,service_level,float64,731
4,A_Daily.csv,abandon_rate,float64,731
5,A_Interval.csv,month,float64,4084
6,A_Interval.csv,day,float64,1
7,A_Interval.csv,interval,float64,4084
8,A_Interval.csv,service_level,float64,4084
9,A_Interval.csv,call_volume,float64,89


In [95]:
phase2_summary_df.to_csv(phase2_dir / "phase2_file_structure_summary.csv", index=False)
phase2_dtype_summary_df.to_csv(phase2_dir / "phase2_dtype_summary.csv", index=False)

print("Phase 2 standardized files saved in:")
print(phase2_dir)
print(sorted([p.name for p in phase2_dir.iterdir()]))

Phase 2 standardized files saved in:
/home/sagemaker-user/Data_Cleaned/Phase2_Standardized
['A_Daily.csv', 'A_Interval.csv', 'B_Daily.csv', 'B_Interval.csv', 'C_Daily.csv', 'C_Interval.csv', 'D_Daily.csv', 'D_Interval.csv', 'Daily_Staffing.csv', 'phase2_dtype_summary.csv', 'phase2_file_structure_summary.csv']


In [96]:
base_dir = Path.cwd()
phase2_dir = base_dir / "Data_Cleaned" / "Phase2_Standardized"

print("Phase 2 directory:", phase2_dir)
print("Exists:", phase2_dir.exists())


phase2_files = [
    "A_Daily.csv",
    "A_Interval.csv",
    "B_Daily.csv",
    "B_Interval.csv",
    "C_Daily.csv",
    "C_Interval.csv",
    "D_Daily.csv",
    "D_Interval.csv",
    "Daily_Staffing.csv"
]

clean_data = {}

for file_name in phase2_files:
    path = phase2_dir / file_name
    if path.exists():
        clean_data[file_name] = pd.read_csv(path)
        print(f"{file_name} loaded | shape = {clean_data[file_name].shape}")
    else:
        print(f"{file_name} not found")

Phase 2 directory: /home/sagemaker-user/Data_Cleaned/Phase2_Standardized
Exists: True
A_Daily.csv loaded | shape = (731, 5)
A_Interval.csv loaded | shape = (4084, 8)
B_Daily.csv loaded | shape = (731, 5)
B_Interval.csv loaded | shape = (4293, 8)
C_Daily.csv loaded | shape = (731, 5)
C_Interval.csv loaded | shape = (4368, 8)
D_Daily.csv loaded | shape = (731, 5)
D_Interval.csv loaded | shape = (4358, 8)
Daily_Staffing.csv loaded | shape = (365, 5)


In [97]:
daily_files = [f for f in clean_data if "_Daily.csv" in f]
interval_files = [f for f in clean_data if "_Interval.csv" in f]
staffing_file = "Daily_Staffing.csv"

print("Daily files:", daily_files)
print("Interval files:", interval_files)
print("Staffing file present:", staffing_file in clean_data)

Daily files: ['A_Daily.csv', 'B_Daily.csv', 'C_Daily.csv', 'D_Daily.csv']
Interval files: ['A_Interval.csv', 'B_Interval.csv', 'C_Interval.csv', 'D_Interval.csv']
Staffing file present: True


In [101]:
def parse_date_column(series):
    """
    Try a few common date formats first, then fall back to generic parsing.
    """
    series = series.astype(str).str.strip()

    common_formats = [
        "%m/%d/%Y",
        "%Y-%m-%d",
        "%m-%d-%Y",
        "%d-%m-%Y"
    ]

    parsed = pd.Series(pd.NaT, index=series.index)

    for fmt in common_formats:
        temp = pd.to_datetime(series, format=fmt, errors="coerce")
        parsed = parsed.fillna(temp)

    # final fallback for anything still not parsed
    still_missing = parsed.isna()
    if still_missing.any():
        parsed.loc[still_missing] = pd.to_datetime(
            series.loc[still_missing],
            errors="coerce"
        )

    return parsed

In [102]:
daily_cleaned = {}
daily_diagnostics = []

for file_name in daily_files:
    df = clean_data[file_name].copy()
    portfolio = get_portfolio(file_name)

    # parse date more carefully
    df["date"] = parse_date_column(df["date"])

    # sort
    df = df.sort_values("date").reset_index(drop=True)

    # flags for time splits
    df["is_train_period"] = df["date"] <= pd.Timestamp("2025-07-31")
    df["is_aug_holdout"] = (
        (df["date"] >= pd.Timestamp("2025-08-01")) &
        (df["date"] <= pd.Timestamp("2025-08-31"))
    )
    df["is_post_aug"] = df["date"] > pd.Timestamp("2025-08-31")

    # missing flags for important columns
    important_cols = ["call_volume", "cct", "service_level", "abandon_rate"]

    for col in important_cols:
        if col in df.columns:
            df[f"{col}_missing_flag"] = df[col].isna()

    # validity flags
    if "call_volume" in df.columns:
        df["call_volume_negative_flag"] = (~df["call_volume"].isna()) & (df["call_volume"] < 0)

    if "cct" in df.columns:
        df["cct_negative_flag"] = (~df["cct"].isna()) & (df["cct"] < 0)

    if "abandon_rate" in df.columns:
        df["abandon_rate_invalid_flag"] = safe_rate_flag(df["abandon_rate"], 0, 100)

    if "service_level" in df.columns:
        df["service_level_invalid_flag"] = safe_rate_flag(df["service_level"], 0, 100)

    # duplicate date flag
    df["duplicate_date_flag"] = df["date"].duplicated(keep=False)

    # continuity check
    exp_range = expected_daily_range(df, "date")
    actual_dates = pd.Series(df["date"].dropna().unique()).sort_values()

    missing_dates_from_sequence = []
    if exp_range is not None:
        missing_dates_from_sequence = list(set(exp_range) - set(actual_dates))

    # missing run diagnostics
    for col in important_cols:
        if col in df.columns:
            runs = find_missing_runs(df[col])

            if len(runs) == 0:
                daily_diagnostics.append({
                    "filename": file_name,
                    "portfolio": portfolio,
                    "metric": col,
                    "missing_run_count": 0,
                    "max_missing_run_length": 0
                })
            else:
                daily_diagnostics.append({
                    "filename": file_name,
                    "portfolio": portfolio,
                    "metric": col,
                    "missing_run_count": len(runs),
                    "max_missing_run_length": max(r[2] for r in runs)
                })

    # general summary row
    daily_diagnostics.append({
        "filename": file_name,
        "portfolio": portfolio,
        "metric": "ALL",
        "row_count": len(df),
        "duplicate_dates": int(df["duplicate_date_flag"].sum()),
        "invalid_dates": int(df["date"].isna().sum()),
        "missing_dates_in_sequence": len(missing_dates_from_sequence),
        "train_rows": int(df["is_train_period"].sum()),
        "aug_holdout_rows": int(df["is_aug_holdout"].sum()),
        "post_aug_rows": int(df["is_post_aug"].sum())
    })

    daily_cleaned[file_name] = df
    print(f"{file_name} cleaned structurally | shape = {df.shape}")

A_Daily.csv cleaned structurally | shape = (731, 17)
B_Daily.csv cleaned structurally | shape = (731, 17)
C_Daily.csv cleaned structurally | shape = (731, 17)
D_Daily.csv cleaned structurally | shape = (731, 17)


/tmp/ipykernel_14136/1621617768.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed.loc[still_missing] = pd.to_datetime(
/tmp/ipykernel_14136/1621617768.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed.loc[still_missing] = pd.to_datetime(
/tmp/ipykernel_14136/1621617768.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed.loc[still_missing] = pd.to_datetime(
/tmp/ipykernel_14136/1621617768.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expec

In [103]:
for file_name in daily_files:
    df = clean_data[file_name].copy()

    print("\n" + "=" * 70)
    print(file_name)
    print("=" * 70)

    print("Sample raw date values:")
    print(df["date"].head(10).tolist())

    print("\nA few unique raw date values:")
    print(df["date"].dropna().astype(str).head(10).tolist())


A_Daily.csv
Sample raw date values:
['01/01/24 Mon', '01/02/24 Tue', '01/03/24 Wed', '01/04/24 Thu', '01/05/24 Fri', '01/06/24 Sat', '01/07/24 Sun', '01/08/24 Mon', '01/09/24 Tue', '01/10/24 Wed']

A few unique raw date values:
['01/01/24 Mon', '01/02/24 Tue', '01/03/24 Wed', '01/04/24 Thu', '01/05/24 Fri', '01/06/24 Sat', '01/07/24 Sun', '01/08/24 Mon', '01/09/24 Tue', '01/10/24 Wed']

B_Daily.csv
Sample raw date values:
['01/01/24 Mon', '01/02/24 Tue', '01/03/24 Wed', '01/04/24 Thu', '01/05/24 Fri', '01/06/24 Sat', '01/07/24 Sun', '01/08/24 Mon', '01/09/24 Tue', '01/10/24 Wed']

A few unique raw date values:
['01/01/24 Mon', '01/02/24 Tue', '01/03/24 Wed', '01/04/24 Thu', '01/05/24 Fri', '01/06/24 Sat', '01/07/24 Sun', '01/08/24 Mon', '01/09/24 Tue', '01/10/24 Wed']

C_Daily.csv
Sample raw date values:
['01/01/24 Mon', '01/02/24 Tue', '01/03/24 Wed', '01/04/24 Thu', '01/05/24 Fri', '01/06/24 Sat', '01/07/24 Sun', '01/08/24 Mon', '01/09/24 Tue', '01/10/24 Wed']

A few unique raw date

In [104]:
def parse_date_column(series):
    # keep only the first token, e.g. "01/01/24" from "01/01/24 Mon"
    date_part = series.astype(str).str.strip().str.split().str[0]

    # parse as month/day/two-digit-year
    parsed = pd.to_datetime(date_part, format="%m/%d/%y", errors="coerce")

    return parsed

In [105]:
daily_cleaned = {}
daily_diagnostics = []

for file_name in daily_files:
    df = clean_data[file_name].copy()
    portfolio = get_portfolio(file_name)

    # parse date
    df["date"] = parse_date_column(df["date"])

    # sort
    df = df.sort_values("date").reset_index(drop=True)

    # time split flags
    df["is_train_period"] = df["date"] <= pd.Timestamp("2025-07-31")
    df["is_aug_holdout"] = (
        (df["date"] >= pd.Timestamp("2025-08-01")) &
        (df["date"] <= pd.Timestamp("2025-08-31"))
    )
    df["is_post_aug"] = df["date"] > pd.Timestamp("2025-08-31")

    # missing flags
    important_cols = ["call_volume", "cct", "service_level", "abandon_rate"]

    for col in important_cols:
        if col in df.columns:
            df[f"{col}_missing_flag"] = df[col].isna()

    # validity flags
    if "call_volume" in df.columns:
        df["call_volume_negative_flag"] = (~df["call_volume"].isna()) & (df["call_volume"] < 0)

    if "cct" in df.columns:
        df["cct_negative_flag"] = (~df["cct"].isna()) & (df["cct"] < 0)

    if "abandon_rate" in df.columns:
        df["abandon_rate_invalid_flag"] = safe_rate_flag(df["abandon_rate"], 0, 100)

    if "service_level" in df.columns:
        df["service_level_invalid_flag"] = safe_rate_flag(df["service_level"], 0, 100)

    # duplicate date flag
    df["duplicate_date_flag"] = df["date"].duplicated(keep=False)

    # continuity check
    exp_range = expected_daily_range(df, "date")
    actual_dates = pd.Series(df["date"].dropna().unique()).sort_values()

    missing_dates_from_sequence = []
    if exp_range is not None:
        missing_dates_from_sequence = list(set(exp_range) - set(actual_dates))

    # missing run diagnostics
    for col in important_cols:
        if col in df.columns:
            runs = find_missing_runs(df[col])

            if len(runs) == 0:
                daily_diagnostics.append({
                    "filename": file_name,
                    "portfolio": portfolio,
                    "metric": col,
                    "missing_run_count": 0,
                    "max_missing_run_length": 0
                })
            else:
                daily_diagnostics.append({
                    "filename": file_name,
                    "portfolio": portfolio,
                    "metric": col,
                    "missing_run_count": len(runs),
                    "max_missing_run_length": max(r[2] for r in runs)
                })

    # summary row
    daily_diagnostics.append({
        "filename": file_name,
        "portfolio": portfolio,
        "metric": "ALL",
        "row_count": len(df),
        "duplicate_dates": int(df["duplicate_date_flag"].sum()),
        "invalid_dates": int(df["date"].isna().sum()),
        "missing_dates_in_sequence": len(missing_dates_from_sequence),
        "train_rows": int(df["is_train_period"].sum()),
        "aug_holdout_rows": int(df["is_aug_holdout"].sum()),
        "post_aug_rows": int(df["is_post_aug"].sum())
    })

    daily_cleaned[file_name] = df
    print(f"{file_name} cleaned structurally | shape = {df.shape}")

A_Daily.csv cleaned structurally | shape = (731, 17)
B_Daily.csv cleaned structurally | shape = (731, 17)
C_Daily.csv cleaned structurally | shape = (731, 17)
D_Daily.csv cleaned structurally | shape = (731, 17)


In [106]:
for file_name in daily_files:
    df = daily_cleaned[file_name]
    print(file_name)
    print("min date:", df["date"].min())
    print("max date:", df["date"].max())
    print("invalid dates:", int(df["date"].isna().sum()))
    print("train rows:", int(df["is_train_period"].sum()))
    print("aug holdout rows:", int(df["is_aug_holdout"].sum()))
    print("post aug rows:", int(df["is_post_aug"].sum()))
    print("-" * 60)

A_Daily.csv
min date: 2024-01-01 00:00:00
max date: 2025-12-31 00:00:00
invalid dates: 0
train rows: 578
aug holdout rows: 31
post aug rows: 122
------------------------------------------------------------
B_Daily.csv
min date: 2024-01-01 00:00:00
max date: 2025-12-31 00:00:00
invalid dates: 0
train rows: 578
aug holdout rows: 31
post aug rows: 122
------------------------------------------------------------
C_Daily.csv
min date: 2024-01-01 00:00:00
max date: 2025-12-31 00:00:00
invalid dates: 0
train rows: 578
aug holdout rows: 31
post aug rows: 122
------------------------------------------------------------
D_Daily.csv
min date: 2024-01-01 00:00:00
max date: 2025-12-31 00:00:00
invalid dates: 0
train rows: 578
aug holdout rows: 31
post aug rows: 122
------------------------------------------------------------


In [107]:
daily_diagnostics_df = pd.DataFrame(daily_diagnostics)
display(daily_diagnostics_df)

,filename,portfolio,metric,missing_run_count,max_missing_run_length,row_count,duplicate_dates,invalid_dates,missing_dates_in_sequence,train_rows,aug_holdout_rows,post_aug_rows
0,A_Daily.csv,A,call_volume,8.0,241.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A_Daily.csv,A,cct,3.0,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A_Daily.csv,A,service_level,1.0,731.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A_Daily.csv,A,abandon_rate,1.0,731.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A_Daily.csv,A,ALL,NaN,NaN,731.0,0.0,0.0,0.0,578.0,31.0,122.0
5,B_Daily.csv,B,call_volume,3.0,364.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,B_Daily.csv,B,cct,3.0,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,B_Daily.csv,B,service_level,1.0,731.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,B_Daily.csv,B,abandon_rate,1.0,731.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,B_Daily.csv,B,ALL,NaN,NaN,731.0,0.0,0.0,0.0,578.0,31.0,122.0


In [108]:
"""
staffing_cleaned = None
staffing_diagnostics = []

if staffing_file in clean_data:
    df = clean_data[staffing_file].copy()

    # parse date
    df["date"] = parse_date_column(df["date"])

    # sort
    df = df.sort_values("date").reset_index(drop=True)

    # date flags
    df["duplicate_date_flag"] = df["date"].duplicated(keep=False)
    df["invalid_date_flag"] = df["date"].isna()

    # identify staffing value columns
    staffing_value_cols = [c for c in df.columns if c != "date"]

    # missing + negative flags
    for col in staffing_value_cols:
        df[f"{col}_missing_flag"] = df[col].isna()

        if pd.api.types.is_numeric_dtype(df[col]):
            df[f"{col}_negative_flag"] = (~df[col].isna()) & (df[col] < 0)

    staffing_diagnostics.append({
        "filename": staffing_file,
        "row_count": len(df),
        "min_date": df["date"].min(),
        "max_date": df["date"].max(),
        "invalid_dates": int(df["invalid_date_flag"].sum()),
        "duplicate_dates": int(df["duplicate_date_flag"].sum())
    })

    staffing_cleaned = df
    print(f"{staffing_file} cleaned structurally | shape = {df.shape}")
"""

KeyError: 'date'

In [109]:
staff_df = clean_data[staffing_file].copy()

print("Staffing columns:")
print(staff_df.columns.tolist())

display(staff_df.head(10))

Staffing columns:
['unnamed_0', 'a', 'b', 'c', 'd']


,unnamed_0,a,b,c,d
0,NaN,47.0,75.0,353.0,143.0
1,NaN,82.0,184.0,491.0,195.0
2,NaN,92.0,186.0,462.0,183.0
3,NaN,70.0,148.0,352.0,155.0
4,NaN,40.0,110.0,224.0,98.0
5,NaN,101.0,190.0,498.0,196.0
6,NaN,NaN,NaN,NaN,188.0
7,NaN,NaN,NaN,NaN,168.0
8,NaN,NaN,NaN,NaN,177.0
9,NaN,92.0,166.0,409.0,179.0


In [110]:
staff_df = clean_data[staffing_file].copy()

print("Shape:", staff_df.shape)
print("Columns:", staff_df.columns.tolist())
display(staff_df.head(10))
display(staff_df.tail(10))

Shape: (365, 5)
Columns: ['unnamed_0', 'a', 'b', 'c', 'd']


,unnamed_0,a,b,c,d
0,NaN,47.0,75.0,353.0,143.0
1,NaN,82.0,184.0,491.0,195.0
2,NaN,92.0,186.0,462.0,183.0
3,NaN,70.0,148.0,352.0,155.0
4,NaN,40.0,110.0,224.0,98.0
5,NaN,101.0,190.0,498.0,196.0
6,NaN,NaN,NaN,NaN,188.0
7,NaN,NaN,NaN,NaN,168.0
8,NaN,NaN,NaN,NaN,177.0
9,NaN,92.0,166.0,409.0,179.0


,unnamed_0,a,b,c,d
355,NaN,NaN,NaN,344.0,159.0
356,NaN,NaN,NaN,332.0,164.0
357,NaN,58.0,107.0,308.0,162.0
358,NaN,60.0,102.0,268.0,135.0
359,NaN,63.0,86.0,284.0,129.0
360,NaN,44.0,68.0,196.0,95.0
361,NaN,30.0,51.0,NaN,NaN
362,NaN,75.0,106.0,NaN,NaN
363,NaN,71.0,108.0,NaN,NaN
364,NaN,64.0,109.0,NaN,NaN


In [111]:
staffing_cleaned = None
staffing_diagnostics = []

if staffing_file in clean_data:
    df = clean_data[staffing_file].copy()

    # rename columns properly
    rename_map = {
        "unnamed_0": "date",
        "a": "staffing_a",
        "b": "staffing_b",
        "c": "staffing_c",
        "d": "staffing_d"
    }
    df = df.rename(columns=rename_map)

    print("Staffing columns after rename:")
    print(df.columns.tolist())

    # parse date
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # sort
    df = df.sort_values("date").reset_index(drop=True)

    # date flags
    df["duplicate_date_flag"] = df["date"].duplicated(keep=False)
    df["invalid_date_flag"] = df["date"].isna()

    # staffing metric columns
    staffing_value_cols = ["staffing_a", "staffing_b", "staffing_c", "staffing_d"]

    # missing and negative flags
    for col in staffing_value_cols:
        if col in df.columns:
            df[f"{col}_missing_flag"] = df[col].isna()
            df[f"{col}_negative_flag"] = (~df[col].isna()) & (df[col] < 0)

    staffing_diagnostics.append({
        "filename": staffing_file,
        "row_count": len(df),
        "min_date": df["date"].min(),
        "max_date": df["date"].max(),
        "invalid_dates": int(df["invalid_date_flag"].sum()),
        "duplicate_dates": int(df["duplicate_date_flag"].sum())
    })

    staffing_cleaned = df
    print(f"{staffing_file} cleaned structurally | shape = {df.shape}")

Staffing columns after rename:
['date', 'staffing_a', 'staffing_b', 'staffing_c', 'staffing_d']
Daily_Staffing.csv cleaned structurally | shape = (365, 15)


In [112]:
display(staffing_cleaned.head(10))
display(staffing_cleaned.tail(10))
display(pd.DataFrame(staffing_diagnostics))

,date,staffing_a,staffing_b,staffing_c,staffing_d,duplicate_date_flag,invalid_date_flag,staffing_a_missing_flag,staffing_a_negative_flag,staffing_b_missing_flag,staffing_b_negative_flag,staffing_c_missing_flag,staffing_c_negative_flag,staffing_d_missing_flag,staffing_d_negative_flag
0,NaT,47.0,75.0,353.0,143.0,True,True,False,False,False,False,False,False,False,False
1,NaT,82.0,184.0,491.0,195.0,True,True,False,False,False,False,False,False,False,False
2,NaT,92.0,186.0,462.0,183.0,True,True,False,False,False,False,False,False,False,False
3,NaT,70.0,148.0,352.0,155.0,True,True,False,False,False,False,False,False,False,False
4,NaT,40.0,110.0,224.0,98.0,True,True,False,False,False,False,False,False,False,False
5,NaT,101.0,190.0,498.0,196.0,True,True,False,False,False,False,False,False,False,False
6,NaT,NaN,NaN,NaN,188.0,True,True,True,False,True,False,True,False,False,False
7,NaT,NaN,NaN,NaN,168.0,True,True,True,False,True,False,True,False,False,False
8,NaT,NaN,NaN,NaN,177.0,True,True,True,False,True,False,True,False,False,False
9,NaT,92.0,166.0,409.0,179.0,True,True,False,False,False,False,False,False,False,False


,date,staffing_a,staffing_b,staffing_c,staffing_d,duplicate_date_flag,invalid_date_flag,staffing_a_missing_flag,staffing_a_negative_flag,staffing_b_missing_flag,staffing_b_negative_flag,staffing_c_missing_flag,staffing_c_negative_flag,staffing_d_missing_flag,staffing_d_negative_flag
355,NaT,NaN,NaN,344.0,159.0,True,True,True,False,True,False,False,False,False,False
356,NaT,NaN,NaN,332.0,164.0,True,True,True,False,True,False,False,False,False,False
357,NaT,58.0,107.0,308.0,162.0,True,True,False,False,False,False,False,False,False,False
358,NaT,60.0,102.0,268.0,135.0,True,True,False,False,False,False,False,False,False,False
359,NaT,63.0,86.0,284.0,129.0,True,True,False,False,False,False,False,False,False,False
360,NaT,44.0,68.0,196.0,95.0,True,True,False,False,False,False,False,False,False,False
361,NaT,30.0,51.0,NaN,NaN,True,True,False,False,False,False,True,False,True,False
362,NaT,75.0,106.0,NaN,NaN,True,True,False,False,False,False,True,False,True,False
363,NaT,71.0,108.0,NaN,NaN,True,True,False,False,False,False,True,False,True,False
364,NaT,64.0,109.0,NaN,NaN,True,True,False,False,False,False,True,False,True,False


,filename,row_count,min_date,max_date,invalid_dates,duplicate_dates
0,Daily_Staffing.csv,365,NaT,NaT,365,365


In [113]:
print("Missing values by staffing column:")
display(
    staffing_cleaned[["staffing_a", "staffing_b", "staffing_c", "staffing_d"]].isna().sum()
)

print("\nNegative values by staffing column:")
for col in ["staffing_a", "staffing_b", "staffing_c", "staffing_d"]:
    print(col, "->", int((staffing_cleaned[col] < 0).sum()))

Missing values by staffing column:


staffing_a    27
staffing_b    35
staffing_c    37
staffing_d    36
dtype: int64


Negative values by staffing column:
staffing_a -> 0
staffing_b -> 0
staffing_c -> 0
staffing_d -> 0


In [114]:
pd.to_datetime(df["date"], errors="coerce")

0     NaT
1     NaT
2     NaT
3     NaT
4     NaT
       ..
360   NaT
361   NaT
362   NaT
363   NaT
364   NaT
Name: date, Length: 365, dtype: datetime64[ns]

In [115]:
staffing_cleaned = None
staffing_diagnostics = []

if staffing_file in clean_data:
    df = clean_data[staffing_file].copy()

    # rename columns based on the real original structure
    rename_map = {
        "unnamed_0": "date",
        "a": "staffing_a",
        "b": "staffing_b",
        "c": "staffing_c",
        "d": "staffing_d"
    }
    df = df.rename(columns=rename_map)

    print("Staffing columns after rename:")
    print(df.columns.tolist())

    # parse date
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # sort
    df = df.sort_values("date").reset_index(drop=True)

    # date flags
    df["duplicate_date_flag"] = df["date"].duplicated(keep=False)
    df["invalid_date_flag"] = df["date"].isna()

    # staffing metric columns
    staffing_value_cols = ["staffing_a", "staffing_b", "staffing_c", "staffing_d"]

    # missing and negative flags
    for col in staffing_value_cols:
        if col in df.columns:
            df[f"{col}_missing_flag"] = df[col].isna()
            df[f"{col}_negative_flag"] = (~df[col].isna()) & (df[col] < 0)

    staffing_diagnostics.append({
        "filename": staffing_file,
        "row_count": len(df),
        "min_date": df["date"].min(),
        "max_date": df["date"].max(),
        "invalid_dates": int(df["invalid_date_flag"].sum()),
        "duplicate_dates": int(df["duplicate_date_flag"].sum())
    })

    staffing_cleaned = df
    print(f"{staffing_file} cleaned structurally | shape = {df.shape}")

Staffing columns after rename:
['date', 'staffing_a', 'staffing_b', 'staffing_c', 'staffing_d']
Daily_Staffing.csv cleaned structurally | shape = (365, 15)


In [116]:
display(staffing_cleaned.head(10))
display(staffing_cleaned.tail(10))
display(pd.DataFrame(staffing_diagnostics))

,date,staffing_a,staffing_b,staffing_c,staffing_d,duplicate_date_flag,invalid_date_flag,staffing_a_missing_flag,staffing_a_negative_flag,staffing_b_missing_flag,staffing_b_negative_flag,staffing_c_missing_flag,staffing_c_negative_flag,staffing_d_missing_flag,staffing_d_negative_flag
0,NaT,47.0,75.0,353.0,143.0,True,True,False,False,False,False,False,False,False,False
1,NaT,82.0,184.0,491.0,195.0,True,True,False,False,False,False,False,False,False,False
2,NaT,92.0,186.0,462.0,183.0,True,True,False,False,False,False,False,False,False,False
3,NaT,70.0,148.0,352.0,155.0,True,True,False,False,False,False,False,False,False,False
4,NaT,40.0,110.0,224.0,98.0,True,True,False,False,False,False,False,False,False,False
5,NaT,101.0,190.0,498.0,196.0,True,True,False,False,False,False,False,False,False,False
6,NaT,NaN,NaN,NaN,188.0,True,True,True,False,True,False,True,False,False,False
7,NaT,NaN,NaN,NaN,168.0,True,True,True,False,True,False,True,False,False,False
8,NaT,NaN,NaN,NaN,177.0,True,True,True,False,True,False,True,False,False,False
9,NaT,92.0,166.0,409.0,179.0,True,True,False,False,False,False,False,False,False,False


,date,staffing_a,staffing_b,staffing_c,staffing_d,duplicate_date_flag,invalid_date_flag,staffing_a_missing_flag,staffing_a_negative_flag,staffing_b_missing_flag,staffing_b_negative_flag,staffing_c_missing_flag,staffing_c_negative_flag,staffing_d_missing_flag,staffing_d_negative_flag
355,NaT,NaN,NaN,344.0,159.0,True,True,True,False,True,False,False,False,False,False
356,NaT,NaN,NaN,332.0,164.0,True,True,True,False,True,False,False,False,False,False
357,NaT,58.0,107.0,308.0,162.0,True,True,False,False,False,False,False,False,False,False
358,NaT,60.0,102.0,268.0,135.0,True,True,False,False,False,False,False,False,False,False
359,NaT,63.0,86.0,284.0,129.0,True,True,False,False,False,False,False,False,False,False
360,NaT,44.0,68.0,196.0,95.0,True,True,False,False,False,False,False,False,False,False
361,NaT,30.0,51.0,NaN,NaN,True,True,False,False,False,False,True,False,True,False
362,NaT,75.0,106.0,NaN,NaN,True,True,False,False,False,False,True,False,True,False
363,NaT,71.0,108.0,NaN,NaN,True,True,False,False,False,False,True,False,True,False
364,NaT,64.0,109.0,NaN,NaN,True,True,False,False,False,False,True,False,True,False


,filename,row_count,min_date,max_date,invalid_dates,duplicate_dates
0,Daily_Staffing.csv,365,NaT,NaT,365,365


In [117]:
interval_cleaned = {}
interval_day_diagnostics = []

for file_name in interval_files:
    df = clean_data[file_name].copy()
    portfolio = get_portfolio(file_name)

    df["month"] = df["month"].astype(str).str.strip()
    df["day"] = pd.to_numeric(df["day"], errors="coerce")
    df["interval"] = pd.to_numeric(df["interval"], errors="coerce")

    base_days = (
        df[["month", "day"]]
        .dropna()
        .drop_duplicates()
        .sort_values(["month", "day"])
        .reset_index(drop=True)
    )

    full_grid = base_days.merge(
        pd.DataFrame({"interval": np.arange(1, 49)}),
        how="cross"
    )

    merged = full_grid.merge(
        df,
        on=["month", "day", "interval"],
        how="left"
    )

    metric_cols = ["service_level", "call_volume", "abandoned_calls", "abandoned_rate", "cct"]
    metric_cols = [c for c in metric_cols if c in merged.columns]

    merged["slot_missing_flag"] = merged[metric_cols].isna().all(axis=1)

    raw_counts = (
        df.groupby(["month", "day"])["interval"]
        .nunique()
        .reset_index(name="raw_unique_intervals")
    )

    merged = merged.merge(raw_counts, on=["month", "day"], how="left")
    merged["incomplete_day_flag"] = merged["raw_unique_intervals"] < 48

    merged["interval_start_hour"] = (merged["interval"] - 1) / 2
    merged["overnight_flag"] = merged["interval_start_hour"] < 7

    if "call_volume" in merged.columns:
        merged["low_volume_flag"] = (~merged["call_volume"].isna()) & (merged["call_volume"] <= 2)
    else:
        merged["low_volume_flag"] = False

    interval_cleaned[file_name] = merged

    interval_day_diagnostics.append({
        "filename": file_name,
        "portfolio": portfolio,
        "rows_after_full_grid": len(merged),
        "unique_days": merged[["month", "day"]].drop_duplicates().shape[0],
        "incomplete_days": int(
            merged[["month", "day", "incomplete_day_flag"]]
            .drop_duplicates()["incomplete_day_flag"]
            .sum()
        ),
        "slot_missing_rows": int(merged["slot_missing_flag"].sum())
    })

    print(f"{file_name} full-grid reconstruction done | shape = {merged.shape}")

A_Interval.csv full-grid reconstruction done | shape = (1488, 14)
B_Interval.csv full-grid reconstruction done | shape = (1488, 14)
C_Interval.csv full-grid reconstruction done | shape = (1488, 14)
D_Interval.csv full-grid reconstruction done | shape = (1488, 14)


In [118]:
interval_day_diagnostics_df = pd.DataFrame(interval_day_diagnostics)
display(interval_day_diagnostics_df)
display(interval_cleaned["A_Interval.csv"].head(20))

,filename,portfolio,rows_after_full_grid,unique_days,incomplete_days,slot_missing_rows
0,A_Interval.csv,A,1488,31,31,1488
1,B_Interval.csv,B,1488,31,31,1488
2,C_Interval.csv,C,1488,31,31,1488
3,D_Interval.csv,D,1488,31,31,1488


,month,day,interval,service_level,call_volume,abandoned_calls,abandoned_rate,cct,slot_missing_flag,raw_unique_intervals,incomplete_day_flag,interval_start_hour,overnight_flag,low_volume_flag
0,nan,1.0,1,NaN,NaN,NaN,NaN,NaN,True,0,True,0.0,True,False
1,nan,1.0,2,NaN,NaN,NaN,NaN,NaN,True,0,True,0.5,True,False
2,nan,1.0,3,NaN,NaN,NaN,NaN,NaN,True,0,True,1.0,True,False
3,nan,1.0,4,NaN,NaN,NaN,NaN,NaN,True,0,True,1.5,True,False
4,nan,1.0,5,NaN,NaN,NaN,NaN,NaN,True,0,True,2.0,True,False
5,nan,1.0,6,NaN,NaN,NaN,NaN,NaN,True,0,True,2.5,True,False
6,nan,1.0,7,NaN,NaN,NaN,NaN,NaN,True,0,True,3.0,True,False
7,nan,1.0,8,NaN,NaN,NaN,NaN,NaN,True,0,True,3.5,True,False
8,nan,1.0,9,NaN,NaN,NaN,NaN,NaN,True,0,True,4.0,True,False
9,nan,1.0,10,NaN,NaN,NaN,NaN,NaN,True,0,True,4.5,True,False


In [119]:
interval_flagged = {}
interval_rule_diagnostics = []

for file_name, df in interval_cleaned.items():
    temp = df.copy()
    portfolio = get_portfolio(file_name)

    if "call_volume" in temp.columns:
        temp["call_volume_negative_flag"] = (~temp["call_volume"].isna()) & (temp["call_volume"] < 0)

    if "abandoned_calls" in temp.columns:
        temp["abandoned_calls_negative_flag"] = (~temp["abandoned_calls"].isna()) & (temp["abandoned_calls"] < 0)

    if "cct" in temp.columns:
        temp["cct_negative_flag"] = (~temp["cct"].isna()) & (temp["cct"] < 0)

    if "abandoned_rate" in temp.columns:
        temp["abandoned_rate_invalid_flag"] = safe_rate_flag(temp["abandoned_rate"], 0, 100)

    if "service_level" in temp.columns:
        temp["service_level_invalid_flag"] = safe_rate_flag(temp["service_level"], 0, 100)

    if "cct" in temp.columns:
        overnight_cct = temp.loc[temp["overnight_flag"] & temp["cct"].notna(), "cct"]

        if len(overnight_cct) > 0:
            q1 = overnight_cct.quantile(0.25)
            q3 = overnight_cct.quantile(0.75)
            iqr = q3 - q1
            extreme_threshold = q3 + 3 * iqr
        else:
            extreme_threshold = np.nan

        temp["extreme_cct_flag"] = (
            temp["overnight_flag"] &
            temp["cct"].notna() &
            (temp["cct"] > extreme_threshold)
        )
    else:
        temp["extreme_cct_flag"] = False

    if "cct" in temp.columns and "call_volume" in temp.columns:
        temp["overnight_lowvol_extreme_cct_flag"] = (
            temp["overnight_flag"] &
            temp["low_volume_flag"] &
            temp["extreme_cct_flag"]
        )
    else:
        temp["overnight_lowvol_extreme_cct_flag"] = False

    if {"abandoned_calls", "abandoned_rate", "call_volume"}.issubset(temp.columns):
        expected_abandoned = temp["call_volume"] * (temp["abandoned_rate"] / 100.0)
        temp["abandoned_consistency_gap"] = (temp["abandoned_calls"] - expected_abandoned).abs()

        temp["abandoned_metric_mismatch_flag"] = (
            temp["abandoned_calls"].notna() &
            temp["abandoned_rate"].notna() &
            temp["call_volume"].notna() &
            (temp["abandoned_consistency_gap"] > 1.0)
        )
    else:
        temp["abandoned_consistency_gap"] = np.nan
        temp["abandoned_metric_mismatch_flag"] = False

    interval_flagged[file_name] = temp

    interval_rule_diagnostics.append({
        "filename": file_name,
        "portfolio": portfolio,
        "negative_call_volume_rows": int(temp.get("call_volume_negative_flag", pd.Series(False)).sum()),
        "negative_abandoned_calls_rows": int(temp.get("abandoned_calls_negative_flag", pd.Series(False)).sum()),
        "negative_cct_rows": int(temp.get("cct_negative_flag", pd.Series(False)).sum()),
        "invalid_abandoned_rate_rows": int(temp.get("abandoned_rate_invalid_flag", pd.Series(False)).sum()),
        "invalid_service_level_rows": int(temp.get("service_level_invalid_flag", pd.Series(False)).sum()),
        "extreme_cct_rows": int(temp["extreme_cct_flag"].sum()),
        "overnight_lowvol_extreme_cct_rows": int(temp["overnight_lowvol_extreme_cct_flag"].sum()),
        "abandoned_metric_mismatch_rows": int(temp["abandoned_metric_mismatch_flag"].sum())
    })

    print(f"{file_name} rule/extreme flags added")

A_Interval.csv rule/extreme flags added
B_Interval.csv rule/extreme flags added
C_Interval.csv rule/extreme flags added
D_Interval.csv rule/extreme flags added


In [120]:
interval_rule_diagnostics_df = pd.DataFrame(interval_rule_diagnostics)
display(interval_rule_diagnostics_df)

,filename,portfolio,negative_call_volume_rows,negative_abandoned_calls_rows,negative_cct_rows,invalid_abandoned_rate_rows,invalid_service_level_rows,extreme_cct_rows,overnight_lowvol_extreme_cct_rows,abandoned_metric_mismatch_rows
0,A_Interval.csv,A,0,0,0,0,0,0,0,0
1,B_Interval.csv,B,0,0,0,0,0,0,0,0
2,C_Interval.csv,C,0,0,0,0,0,0,0,0
3,D_Interval.csv,D,0,0,0,0,0,0,0,0


In [121]:
daily_profile_rows = []

for file_name, df in daily_cleaned.items():
    temp = df.copy()
    portfolio = get_portfolio(file_name)

    temp["weekday_num"] = temp["date"].dt.weekday
    temp["weekday_name"] = temp["date"].dt.day_name()
    temp["is_weekend"] = temp["weekday_num"] >= 5
    temp["year_month"] = temp["date"].dt.to_period("M").astype(str)

    for metric in ["call_volume", "cct"]:
        if metric in temp.columns:
            summary = (
                temp.groupby(["weekday_name", "is_weekend"], as_index=False)[metric]
                .mean()
            )
            summary["filename"] = file_name
            summary["portfolio"] = portfolio
            summary["metric"] = metric
            daily_profile_rows.append(summary)

daily_weekday_profile_df = pd.concat(daily_profile_rows, ignore_index=True)
display(daily_weekday_profile_df.head(20))

,weekday_name,is_weekend,call_volume,filename,portfolio,metric,cct
0,Friday,False,NaN,A_Daily.csv,A,call_volume,NaN
1,Monday,False,NaN,A_Daily.csv,A,call_volume,NaN
2,Saturday,True,NaN,A_Daily.csv,A,call_volume,NaN
3,Sunday,True,691.666667,A_Daily.csv,A,call_volume,NaN
4,Thursday,False,444.333333,A_Daily.csv,A,call_volume,NaN
5,Tuesday,False,NaN,A_Daily.csv,A,call_volume,NaN
6,Wednesday,False,294.000000,A_Daily.csv,A,call_volume,NaN
7,Friday,False,NaN,A_Daily.csv,A,cct,313.323400
8,Monday,False,NaN,A_Daily.csv,A,cct,315.164257
9,Saturday,True,NaN,A_Daily.csv,A,cct,315.587300


In [124]:
daily_missing_weekday = []

for file_name, df in daily_cleaned.items():
    temp = df.copy()
    portfolio = get_portfolio(file_name)

    temp["weekday_name"] = temp["date"].dt.day_name()

    for metric in ["call_volume", "cct", "service_level", "abandon_rate"]:
        if metric in temp.columns:
            miss = (
                temp.groupby("weekday_name", as_index=False)
                .agg(missing_count=(metric, lambda x: x.isna().sum()))
            )

            miss["filename"] = file_name
            miss["portfolio"] = portfolio
            miss["metric"] = metric

            daily_missing_weekday.append(miss)

daily_missing_weekday_df = pd.concat(daily_missing_weekday, ignore_index=True)
display(daily_missing_weekday_df.head(20))

,weekday_name,missing_count,filename,portfolio,metric
0,Friday,104,A_Daily.csv,A,call_volume
1,Monday,105,A_Daily.csv,A,call_volume
2,Saturday,104,A_Daily.csv,A,call_volume
3,Sunday,101,A_Daily.csv,A,call_volume
4,Thursday,101,A_Daily.csv,A,call_volume
5,Tuesday,105,A_Daily.csv,A,call_volume
6,Wednesday,104,A_Daily.csv,A,call_volume
7,Friday,4,A_Daily.csv,A,cct
8,Monday,4,A_Daily.csv,A,cct
9,Saturday,4,A_Daily.csv,A,cct


In [125]:
interval_profile_rows = []

for file_name, df in interval_cleaned.items():
    temp = df.copy()
    portfolio = get_portfolio(file_name)

    temp["is_weekend_proxy"] = False  # we do not yet have full calendar dates here
    temp["daypart"] = np.where(temp["interval_start_hour"] < 7, "overnight", "daytime")

    for metric in ["call_volume", "cct", "abandoned_rate"]:
        if metric in temp.columns:
            summary = (
                temp.groupby(["interval", "daypart"], as_index=False)[metric]
                .mean()
            )
            summary["filename"] = file_name
            summary["portfolio"] = portfolio
            summary["metric"] = metric
            interval_profile_rows.append(summary)

interval_profile_df = pd.concat(interval_profile_rows, ignore_index=True)
display(interval_profile_df.head(30))

,interval,daypart,call_volume,filename,portfolio,metric,cct,abandoned_rate
0,1,overnight,NaN,A_Interval.csv,A,call_volume,NaN,NaN
1,2,overnight,NaN,A_Interval.csv,A,call_volume,NaN,NaN
2,3,overnight,NaN,A_Interval.csv,A,call_volume,NaN,NaN
3,4,overnight,NaN,A_Interval.csv,A,call_volume,NaN,NaN
4,5,overnight,NaN,A_Interval.csv,A,call_volume,NaN,NaN
5,6,overnight,NaN,A_Interval.csv,A,call_volume,NaN,NaN
6,7,overnight,NaN,A_Interval.csv,A,call_volume,NaN,NaN
7,8,overnight,NaN,A_Interval.csv,A,call_volume,NaN,NaN
8,9,overnight,NaN,A_Interval.csv,A,call_volume,NaN,NaN
9,10,overnight,NaN,A_Interval.csv,A,call_volume,NaN,NaN


In [126]:
slot_missing_profile = []

for file_name, df in interval_cleaned.items():
    temp = df.copy()
    portfolio = get_portfolio(file_name)

    summary = (
        temp.groupby("interval", as_index=False)["slot_missing_flag"]
        .sum()
        .rename(columns={"slot_missing_flag": "missing_slot_count"})
    )
    summary["filename"] = file_name
    summary["portfolio"] = portfolio
    slot_missing_profile.append(summary)

slot_missing_profile_df = pd.concat(slot_missing_profile, ignore_index=True)
display(slot_missing_profile_df.head(30))

,interval,missing_slot_count,filename,portfolio
0,1,31,A_Interval.csv,A
1,2,31,A_Interval.csv,A
2,3,31,A_Interval.csv,A
3,4,31,A_Interval.csv,A
4,5,31,A_Interval.csv,A
5,6,31,A_Interval.csv,A
6,7,31,A_Interval.csv,A
7,8,31,A_Interval.csv,A
8,9,31,A_Interval.csv,A
9,10,31,A_Interval.csv,A


In [127]:
display(interval_day_diagnostics_df)

,filename,portfolio,rows_after_full_grid,unique_days,incomplete_days,slot_missing_rows
0,A_Interval.csv,A,1488,31,31,1488
1,B_Interval.csv,B,1488,31,31,1488
2,C_Interval.csv,C,1488,31,31,1488
3,D_Interval.csv,D,1488,31,31,1488


In [128]:
if staffing_cleaned is not None:
    staffing_profile = staffing_cleaned.copy()
    staffing_profile["weekday_name"] = staffing_profile["date"].dt.day_name()
    staffing_profile["is_weekend"] = staffing_profile["date"].dt.weekday >= 5

    staffing_summary = (
        staffing_profile.groupby(["weekday_name", "is_weekend"], as_index=False)[
            ["staffing_a", "staffing_b", "staffing_c", "staffing_d"]
        ].mean()
    )

    display(staffing_summary)

,weekday_name,is_weekend,staffing_a,staffing_b,staffing_c,staffing_d


In [129]:
print("Invalid staffing dates:", staffing_cleaned["date"].isna().sum())
print("Min date:", staffing_cleaned["date"].min())
print("Max date:", staffing_cleaned["date"].max())

display(staffing_cleaned[["date", "staffing_a", "staffing_b", "staffing_c", "staffing_d"]].head(10))

Invalid staffing dates: 365
Min date: NaT
Max date: NaT


,date,staffing_a,staffing_b,staffing_c,staffing_d
0,NaT,47.0,75.0,353.0,143.0
1,NaT,82.0,184.0,491.0,195.0
2,NaT,92.0,186.0,462.0,183.0
3,NaT,70.0,148.0,352.0,155.0
4,NaT,40.0,110.0,224.0,98.0
5,NaT,101.0,190.0,498.0,196.0
6,NaT,NaN,NaN,NaN,188.0
7,NaT,NaN,NaN,NaN,168.0
8,NaT,NaN,NaN,NaN,177.0
9,NaT,92.0,166.0,409.0,179.0


In [134]:
staffing_cleaned = None
staffing_diagnostics = []

if staffing_file in clean_data:
    df = clean_data[staffing_file].copy()

    rename_map = {
        "unnamed_0": "date",
        "a": "staffing_a",
        "b": "staffing_b",
        "c": "staffing_c",
        "d": "staffing_d"
    }
    df = df.rename(columns=rename_map)

    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    df = df.sort_values("date").reset_index(drop=True)

    df["duplicate_date_flag"] = df["date"].duplicated(keep=False)
    df["invalid_date_flag"] = df["date"].isna()

    for col in ["staffing_a", "staffing_b", "staffing_c", "staffing_d"]:
        df[f"{col}_missing_flag"] = df[col].isna()
        df[f"{col}_negative_flag"] = (~df[col].isna()) & (df[col] < 0)

    staffing_diagnostics.append({
        "filename": staffing_file,
        "row_count": len(df),
        "min_date": df["date"].min(),
        "max_date": df["date"].max(),
        "invalid_dates": int(df["invalid_date_flag"].sum()),
        "duplicate_dates": int(df["duplicate_date_flag"].sum())
    })

    staffing_cleaned = df
    print(f"{staffing_file} cleaned structurally | shape = {df.shape}")

Daily_Staffing.csv cleaned structurally | shape = (365, 15)


In [135]:
display(staffing_cleaned.head(10))
display(pd.DataFrame(staffing_diagnostics))

,date,staffing_a,staffing_b,staffing_c,staffing_d,duplicate_date_flag,invalid_date_flag,staffing_a_missing_flag,staffing_a_negative_flag,staffing_b_missing_flag,staffing_b_negative_flag,staffing_c_missing_flag,staffing_c_negative_flag,staffing_d_missing_flag,staffing_d_negative_flag
0,NaT,47.0,75.0,353.0,143.0,True,True,False,False,False,False,False,False,False,False
1,NaT,82.0,184.0,491.0,195.0,True,True,False,False,False,False,False,False,False,False
2,NaT,92.0,186.0,462.0,183.0,True,True,False,False,False,False,False,False,False,False
3,NaT,70.0,148.0,352.0,155.0,True,True,False,False,False,False,False,False,False,False
4,NaT,40.0,110.0,224.0,98.0,True,True,False,False,False,False,False,False,False,False
5,NaT,101.0,190.0,498.0,196.0,True,True,False,False,False,False,False,False,False,False
6,NaT,NaN,NaN,NaN,188.0,True,True,True,False,True,False,True,False,False,False
7,NaT,NaN,NaN,NaN,168.0,True,True,True,False,True,False,True,False,False,False
8,NaT,NaN,NaN,NaN,177.0,True,True,True,False,True,False,True,False,False,False
9,NaT,92.0,166.0,409.0,179.0,True,True,False,False,False,False,False,False,False,False


,filename,row_count,min_date,max_date,invalid_dates,duplicate_dates
0,Daily_Staffing.csv,365,NaT,NaT,365,365


In [136]:
interval_cleaned = {}
interval_day_diagnostics = []

for file_name in interval_files:
    df = clean_data[file_name].copy()
    portfolio = get_portfolio(file_name)

    df["month"] = df["month"].astype(str).str.strip()
    df["day"] = pd.to_numeric(df["day"], errors="coerce")
    df["interval"] = pd.to_numeric(df["interval"], errors="coerce")

    base_days = (
        df[["month", "day"]]
        .dropna()
        .drop_duplicates()
        .sort_values(["month", "day"])
        .reset_index(drop=True)
    )

    full_grid = base_days.merge(
        pd.DataFrame({"interval": np.arange(1, 49)}),
        how="cross"
    )

    merged = full_grid.merge(
        df,
        on=["month", "day", "interval"],
        how="left"
    )

    metric_cols = ["service_level", "call_volume", "abandoned_calls", "abandoned_rate", "cct"]
    metric_cols = [c for c in metric_cols if c in merged.columns]

    merged["slot_missing_flag"] = merged[metric_cols].isna().all(axis=1)

    raw_counts = (
        df.groupby(["month", "day"])["interval"]
        .nunique()
        .reset_index(name="raw_unique_intervals")
    )

    merged = merged.merge(raw_counts, on=["month", "day"], how="left")
    merged["incomplete_day_flag"] = merged["raw_unique_intervals"] < 48

    merged["interval_start_hour"] = (merged["interval"] - 1) / 2
    merged["overnight_flag"] = merged["interval_start_hour"] < 7

    if "call_volume" in merged.columns:
        merged["low_volume_flag"] = (~merged["call_volume"].isna()) & (merged["call_volume"] <= 2)
    else:
        merged["low_volume_flag"] = False

    interval_cleaned[file_name] = merged

    interval_day_diagnostics.append({
        "filename": file_name,
        "portfolio": portfolio,
        "rows_after_full_grid": len(merged),
        "unique_days": merged[["month", "day"]].drop_duplicates().shape[0],
        "incomplete_days": int(
            merged[["month", "day", "incomplete_day_flag"]]
            .drop_duplicates()["incomplete_day_flag"]
            .sum()
        ),
        "slot_missing_rows": int(merged["slot_missing_flag"].sum())
    })

    print(f"{file_name} full-grid reconstruction done | shape = {merged.shape}")

A_Interval.csv full-grid reconstruction done | shape = (1488, 14)
B_Interval.csv full-grid reconstruction done | shape = (1488, 14)
C_Interval.csv full-grid reconstruction done | shape = (1488, 14)
D_Interval.csv full-grid reconstruction done | shape = (1488, 14)


In [137]:
interval_day_diagnostics_df = pd.DataFrame(interval_day_diagnostics)
display(interval_day_diagnostics_df)

,filename,portfolio,rows_after_full_grid,unique_days,incomplete_days,slot_missing_rows
0,A_Interval.csv,A,1488,31,31,1488
1,B_Interval.csv,B,1488,31,31,1488
2,C_Interval.csv,C,1488,31,31,1488
3,D_Interval.csv,D,1488,31,31,1488


In [138]:
interval_flagged = {}
interval_rule_diagnostics = []

for file_name, df in interval_cleaned.items():
    temp = df.copy()
    portfolio = get_portfolio(file_name)

    if "call_volume" in temp.columns:
        temp["call_volume_negative_flag"] = (~temp["call_volume"].isna()) & (temp["call_volume"] < 0)

    if "abandoned_calls" in temp.columns:
        temp["abandoned_calls_negative_flag"] = (~temp["abandoned_calls"].isna()) & (temp["abandoned_calls"] < 0)

    if "cct" in temp.columns:
        temp["cct_negative_flag"] = (~temp["cct"].isna()) & (temp["cct"] < 0)

    if "abandoned_rate" in temp.columns:
        temp["abandoned_rate_invalid_flag"] = safe_rate_flag(temp["abandoned_rate"], 0, 100)

    if "service_level" in temp.columns:
        temp["service_level_invalid_flag"] = safe_rate_flag(temp["service_level"], 0, 100)

    if "cct" in temp.columns:
        overnight_cct = temp.loc[temp["overnight_flag"] & temp["cct"].notna(), "cct"]

        if len(overnight_cct) > 0:
            q1 = overnight_cct.quantile(0.25)
            q3 = overnight_cct.quantile(0.75)
            iqr = q3 - q1
            extreme_threshold = q3 + 3 * iqr
        else:
            extreme_threshold = np.nan

        temp["extreme_cct_flag"] = (
            temp["overnight_flag"] &
            temp["cct"].notna() &
            (temp["cct"] > extreme_threshold)
        )
    else:
        temp["extreme_cct_flag"] = False

    if "cct" in temp.columns and "call_volume" in temp.columns:
        temp["overnight_lowvol_extreme_cct_flag"] = (
            temp["overnight_flag"] &
            temp["low_volume_flag"] &
            temp["extreme_cct_flag"]
        )
    else:
        temp["overnight_lowvol_extreme_cct_flag"] = False

    if {"abandoned_calls", "abandoned_rate", "call_volume"}.issubset(temp.columns):
        expected_abandoned = temp["call_volume"] * (temp["abandoned_rate"] / 100.0)
        temp["abandoned_consistency_gap"] = (temp["abandoned_calls"] - expected_abandoned).abs()

        temp["abandoned_metric_mismatch_flag"] = (
            temp["abandoned_calls"].notna() &
            temp["abandoned_rate"].notna() &
            temp["call_volume"].notna() &
            (temp["abandoned_consistency_gap"] > 1.0)
        )
    else:
        temp["abandoned_consistency_gap"] = np.nan
        temp["abandoned_metric_mismatch_flag"] = False

    interval_flagged[file_name] = temp

    interval_rule_diagnostics.append({
        "filename": file_name,
        "portfolio": portfolio,
        "negative_call_volume_rows": int(temp.get("call_volume_negative_flag", pd.Series(False)).sum()),
        "negative_abandoned_calls_rows": int(temp.get("abandoned_calls_negative_flag", pd.Series(False)).sum()),
        "negative_cct_rows": int(temp.get("cct_negative_flag", pd.Series(False)).sum()),
        "invalid_abandoned_rate_rows": int(temp.get("abandoned_rate_invalid_flag", pd.Series(False)).sum()),
        "invalid_service_level_rows": int(temp.get("service_level_invalid_flag", pd.Series(False)).sum()),
        "extreme_cct_rows": int(temp["extreme_cct_flag"].sum()),
        "overnight_lowvol_extreme_cct_rows": int(temp["overnight_lowvol_extreme_cct_flag"].sum()),
        "abandoned_metric_mismatch_rows": int(temp["abandoned_metric_mismatch_flag"].sum())
    })

    print(f"{file_name} rule/extreme flags added")

A_Interval.csv rule/extreme flags added
B_Interval.csv rule/extreme flags added
C_Interval.csv rule/extreme flags added
D_Interval.csv rule/extreme flags added


In [139]:
interval_rule_diagnostics_df = pd.DataFrame(interval_rule_diagnostics)
display(interval_rule_diagnostics_df)

,filename,portfolio,negative_call_volume_rows,negative_abandoned_calls_rows,negative_cct_rows,invalid_abandoned_rate_rows,invalid_service_level_rows,extreme_cct_rows,overnight_lowvol_extreme_cct_rows,abandoned_metric_mismatch_rows
0,A_Interval.csv,A,0,0,0,0,0,0,0,0
1,B_Interval.csv,B,0,0,0,0,0,0,0,0
2,C_Interval.csv,C,0,0,0,0,0,0,0,0
3,D_Interval.csv,D,0,0,0,0,0,0,0,0


In [140]:
df = clean_data["A_Interval.csv"].copy()

print("Month unique values:", df["month"].unique()[:10])
print("Day dtype:", df["day"].dtype)
print("Interval dtype:", df["interval"].dtype)

display(df[["month", "day", "interval"]].head(10))

Month unique values: [nan]
Day dtype: float64
Interval dtype: float64


,month,day,interval
0,NaN,1.0,NaN
1,NaN,1.0,NaN
2,NaN,1.0,NaN
3,NaN,1.0,NaN
4,NaN,1.0,NaN
5,NaN,1.0,NaN
6,NaN,1.0,NaN
7,NaN,1.0,NaN
8,NaN,1.0,NaN
9,NaN,1.0,NaN


In [141]:
df["month"] = df["month"].astype(str).str.strip()
df["day"] = pd.to_numeric(df["day"], errors="coerce")
df["interval"] = pd.to_numeric(df["interval"], errors="coerce")

print("After cleaning:")
print("Month unique:", df["month"].unique()[:10])
print("Day dtype:", df["day"].dtype)
print("Interval dtype:", df["interval"].dtype)

After cleaning:
Month unique: ['nan']
Day dtype: float64
Interval dtype: float64


In [142]:
# create real date
df["date"] = pd.to_datetime(
    df["month"] + " " + df["day"].astype(str) + " 2025",
    errors="coerce"
)

# check
print("Invalid dates:", df["date"].isna().sum())

base_days = (
    df[["date"]]
    .dropna()
    .drop_duplicates()
    .sort_values("date")
    .reset_index(drop=True)
)

Invalid dates: 4084


/tmp/ipykernel_14136/1548896336.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(


In [143]:
df = clean_data["A_Interval.csv"].copy()

print("Raw columns:", df.columns.tolist())
print("\nMonth unique sample:")
print(df["month"].dropna().astype(str).unique()[:20])

print("\nDay unique sample:")
print(df["day"].dropna().astype(str).unique()[:20])

print("\nInterval unique sample:")
print(df["interval"].dropna().astype(str).unique()[:20])

display(df[["month", "day", "interval"]].head(20))

Raw columns: ['month', 'day', 'interval', 'service_level', 'call_volume', 'abandoned_calls', 'abandoned_rate', 'cct']

Month unique sample:
[]

Day unique sample:
['1.0' '2.0' '3.0' '4.0' '5.0' '6.0' '7.0' '8.0' '9.0' '10.0' '11.0'
 '12.0' '13.0' '14.0' '15.0' '16.0' '17.0' '18.0' '19.0' '20.0']

Interval unique sample:
[]


,month,day,interval
0,NaN,1.0,NaN
1,NaN,1.0,NaN
2,NaN,1.0,NaN
3,NaN,1.0,NaN
4,NaN,1.0,NaN
5,NaN,1.0,NaN
6,NaN,1.0,NaN
7,NaN,1.0,NaN
8,NaN,1.0,NaN
9,NaN,1.0,NaN


In [144]:
def build_interval_date(month_series, day_series, year=2025):
    month_num = pd.to_numeric(month_series, errors="coerce")
    day_num = pd.to_numeric(day_series, errors="coerce")

    date_df = pd.DataFrame({
        "year": year,
        "month": month_num,
        "day": day_num
    })

    return pd.to_datetime(date_df, errors="coerce")

In [145]:
interval_cleaned = {}
interval_day_diagnostics = []

for file_name in interval_files:
    df = clean_data[file_name].copy()
    portfolio = get_portfolio(file_name)

    # clean key columns
    df["month"] = pd.to_numeric(df["month"], errors="coerce")
    df["day"] = pd.to_numeric(df["day"], errors="coerce")
    df["interval"] = pd.to_numeric(df["interval"], errors="coerce")

    # create real date from numeric month/day
    df["date"] = build_interval_date(df["month"], df["day"], year=2025)

    print(f"\n{file_name}")
    print("Invalid dates:", int(df["date"].isna().sum()))

    # keep only rows with usable date + interval
    df_valid = df.dropna(subset=["date", "interval"]).copy()

    # get all available days
    base_days = (
        df_valid[["date"]]
        .drop_duplicates()
        .sort_values("date")
        .reset_index(drop=True)
    )

    # build full 48-slot grid per day
    full_grid = base_days.merge(
        pd.DataFrame({"interval": np.arange(1, 49)}),
        how="cross"
    )

    merged = full_grid.merge(
        df_valid,
        on=["date", "interval"],
        how="left",
        suffixes=("", "_raw")
    )

    # re-create month/day from date for convenience
    merged["month"] = merged["date"].dt.month
    merged["day"] = merged["date"].dt.day

    metric_cols = ["service_level", "call_volume", "abandoned_calls", "abandoned_rate", "cct"]
    metric_cols = [c for c in metric_cols if c in merged.columns]

    # slot missing = no original metric row matched
    merged["slot_missing_flag"] = merged[metric_cols].isna().all(axis=1)

    # raw completeness by day
    raw_counts = (
        df_valid.groupby("date")["interval"]
        .nunique()
        .reset_index(name="raw_unique_intervals")
    )

    merged = merged.merge(raw_counts, on="date", how="left")
    merged["incomplete_day_flag"] = merged["raw_unique_intervals"] < 48

    # time helpers
    merged["interval_start_hour"] = (merged["interval"] - 1) / 2
    merged["overnight_flag"] = merged["interval_start_hour"] < 7

    if "call_volume" in merged.columns:
        merged["low_volume_flag"] = (~merged["call_volume"].isna()) & (merged["call_volume"] <= 2)
    else:
        merged["low_volume_flag"] = False

    interval_cleaned[file_name] = merged

    interval_day_diagnostics.append({
        "filename": file_name,
        "portfolio": portfolio,
        "rows_after_full_grid": len(merged),
        "unique_days": merged["date"].nunique(),
        "incomplete_days": int(
            merged[["date", "incomplete_day_flag"]]
            .drop_duplicates()["incomplete_day_flag"]
            .sum()
        ),
        "slot_missing_rows": int(merged["slot_missing_flag"].sum())
    })

    print(f"{file_name} full-grid reconstruction done | shape = {merged.shape}")


A_Interval.csv
Invalid dates: 4084
A_Interval.csv full-grid reconstruction done | shape = (0, 15)

B_Interval.csv
Invalid dates: 4293
B_Interval.csv full-grid reconstruction done | shape = (0, 15)

C_Interval.csv
Invalid dates: 4368
C_Interval.csv full-grid reconstruction done | shape = (0, 15)

D_Interval.csv
Invalid dates: 4358
D_Interval.csv full-grid reconstruction done | shape = (0, 15)


In [146]:
interval_day_diagnostics_df = pd.DataFrame(interval_day_diagnostics)
display(interval_day_diagnostics_df)

,filename,portfolio,rows_after_full_grid,unique_days,incomplete_days,slot_missing_rows
0,A_Interval.csv,A,0,0,0,0
1,B_Interval.csv,B,0,0,0,0
2,C_Interval.csv,C,0,0,0,0
3,D_Interval.csv,D,0,0,0,0


In [149]:
# clean month
df["month"] = df["month"].astype("string").str.strip()
df.loc[df["month"].isin(["", "nan", "None"]), "month"] = pd.NA
df["month"] = df["month"].ffill()

In [151]:

"""
def normalize_month(s):
    s = s.astype(str).str.strip().str.lower()
    return s.map({
        "april":4, "apr":4, "4":4, "4.0":4,
        "may":5,             "5":5, "5.0":5,
        "june":6, "jun":6,  "6":6, "6.0":6
    })

interval_cleaned = {}
interval_day_diagnostics = []

interval_raw_names = {
    "A_Interval.csv": "A_Interval.csv",
    "B_Interval.csv": "B_Interval.csv",
    "C_Interval.csv": "C_Interval.csv",
    "D_Interval.csv": "D_Interval.csv",
}

for file_name, raw_name in interval_raw_names.items():
    portfolio = get_portfolio(file_name)

    # ── Load from RAW (not clean_data — that has month/interval destroyed) ──
    df = pd.read_csv(raw_dir / raw_name)
    df = standardize_columns(df)   # lowercase cols only

    # ── Step 1: Fix month — ffill BEFORE anything else ──
    df["month"] = df["month"].astype(str).str.strip()
    df.loc[df["month"].isin(["", "nan", "None", "NaN"]), "month"] = np.nan
    df["month"] = df["month"].ffill()
    df["month_num"] = normalize_month(df["month"])

    # ── Step 2: Fix day ──
    df["day"] = pd.to_numeric(df["day"], errors="coerce")

    # ── Step 3: Parse interval HH:MM:SS → slot_index 0–47 ──
    iv = df["interval"].astype(str).str.strip()
    hour   = pd.to_numeric(iv.str[:2],  errors="coerce")
    minute = pd.to_numeric(iv.str[3:5], errors="coerce")
    df["slot_index"] = (hour * 2 + (minute // 30)).astype("Int64")

    # ── Step 4: Build date ──
    df["date"] = pd.to_datetime(
        pd.DataFrame({"year": 2025, "month": df["month_num"], "day": df["day"]}),
        errors="coerce"
    )

    print(f"\n{file_name}")
    print("  Invalid dates:", df["date"].isna().sum())
    print("  Invalid slots:", df["slot_index"].isna().sum())

    # ── Step 5: Build complete 48-slot grid ──
    base_days = (df[["date"]].dropna().drop_duplicates()
                 .sort_values("date").reset_index(drop=True))
    print("  Unique days:", len(base_days))   # expect 91

    full_grid = base_days.merge(
        pd.DataFrame({"slot_index": range(48)}), how="cross"
    )
    print("  Grid size:", len(full_grid))     # expect 4368

    df_valid = df.dropna(subset=["date", "slot_index"]).copy()
    merged = full_grid.merge(df_valid, on=["date", "slot_index"], how="left")

    # ── Step 6: Restore readable columns ──
    merged["month"]               = merged["date"].dt.month
    merged["day"]                 = merged["date"].dt.day
    merged["interval_start_hour"] = merged["slot_index"] // 2
    merged["overnight_flag"]      = merged["interval_start_hour"] < 7

    metric_cols = [c for c in ["service_level","call_volume",
                                "abandoned_calls","abandoned_rate","cct"]
                   if c in merged.columns]
    merged["slot_missing_flag"] = merged[metric_cols].isna().all(axis=1)

    if "call_volume" in merged.columns:
        merged["low_volume_flag"] = merged["call_volume"].fillna(0) <= 2

    interval_cleaned[file_name] = merged

    interval_day_diagnostics.append({
        "filename":           file_name,
        "portfolio":          portfolio,
        "rows_after_full_grid": len(merged),
        "unique_days":        merged["date"].nunique(),
        "slot_missing_rows":  int(merged["slot_missing_flag"].sum()),
    })
    print(f"  ✓ Done | shape = {merged.shape}")

interval_day_diagnostics_df = pd.DataFrame(interval_day_diagnostics)
display(interval_day_diagnostics_df)
"""


A_Interval.csv
  Invalid dates: 1
  Invalid slots: 1536
  Unique days: 91
  Grid size: 4368
  ✓ Done | shape = (4368, 15)

B_Interval.csv
  Invalid dates: 0
  Invalid slots: 1750
  Unique days: 91
  Grid size: 4368
  ✓ Done | shape = (4368, 15)

C_Interval.csv
  Invalid dates: 0
  Invalid slots: 1824
  Unique days: 91
  Grid size: 4368
  ✓ Done | shape = (4368, 15)

D_Interval.csv
  Invalid dates: 0
  Invalid slots: 1810
  Unique days: 91
  Grid size: 4368
  ✓ Done | shape = (4368, 15)


,filename,portfolio,rows_after_full_grid,unique_days,slot_missing_rows
0,A_Interval.csv,A,4368,91,1865
1,B_Interval.csv,B,4368,91,1867
2,C_Interval.csv,C,4368,91,1835
3,D_Interval.csv,D,4368,91,1825


In [155]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [156]:

"""
for file_name, df in daily_cleaned.items():

    # Isolated gaps → time-based interpolation (train only)
    train_mask = df["is_train_period"] == True

    for col in ["call_volume", "cct", "service_level", "abandon_rate"]:
        if col not in df.columns:
            continue

        # Step 1: interpolate isolated gaps
        interpolated = (
            df.loc[train_mask]
            .set_index("date")[col]
            .interpolate(method="time", limit=2)
            .values
        )
        df.loc[train_mask, col] = interpolated

        # Clustered gaps → same-weekday median from ±28 rows
        for idx in df[train_mask & df[col].isna()].index:
            dow    = df.loc[idx, "date"].dayofweek
            window = df.loc[max(0, idx-28) : idx+28]
            peers  = window[(window["date"].dt.dayofweek == dow) & window[col].notna()]
            if not peers.empty:
                df.loc[idx, col] = peers[col].median()

        # Final fallback → global weekday median
        for idx in df[train_mask & df[col].isna()].index:
            dow = df.loc[idx, "date"].dayofweek
            med = df.loc[train_mask & (df["date"].dt.dayofweek == dow), col].median()
            df.loc[idx, col] = med

    daily_cleaned[file_name] = df
    remaining = df.loc[train_mask, ["call_volume","cct","abandon_rate"]].isna().sum().sum()
    print(f"{file_name} | remaining NaN in train set: {remaining}")
"""

A_Daily.csv | remaining NaN in train set: 578
B_Daily.csv | remaining NaN in train set: 744
C_Daily.csv | remaining NaN in train set: 1156
D_Daily.csv | remaining NaN in train set: 1156


In [157]:
"""
for col in ["staffing_a","staffing_b","staffing_c","staffing_d"]:
    staffing_cleaned[col] = staffing_cleaned[col].ffill().bfill()
"""

In [158]:
for file_name, df in daily_cleaned.items():

    df["_dow"]   = df["date"].dt.dayofweek
    df["_month"] = df["date"].dt.month
    train_rows   = df["is_train_period"] == True

    for col in ["call_volume", "cct", "service_level", "abandon_rate"]:
        if col not in df.columns:
            continue

        # Pass 1: (weekday + month) group median
        group_med = (
            df[train_rows]
            .groupby(["_dow", "_month"])[col]
            .transform("median")
        )
        mask = train_rows & df[col].isna()
        df.loc[mask, col] = group_med[mask]

        # Pass 2: weekday-only median fallback
        group_med2 = (
            df[train_rows]
            .groupby("_dow")[col]
            .transform("median")
        )
        mask = train_rows & df[col].isna()
        df.loc[mask, col] = group_med2[mask]

        # Pass 3: global median safety net
        mask = train_rows & df[col].isna()
        df.loc[mask, col] = df.loc[train_rows, col].median()

    df = df.drop(columns=["_dow", "_month"])
    daily_cleaned[file_name] = df

    remaining = df.loc[train_rows, ["call_volume","cct","abandon_rate"]].isna().sum().sum()
    print(f"{file_name} | remaining NaN in train set: {remaining}")

A_Daily.csv | remaining NaN in train set: 578
B_Daily.csv | remaining NaN in train set: 578
C_Daily.csv | remaining NaN in train set: 1156
D_Daily.csv | remaining NaN in train set: 1156


In [161]:
for file_name, df in interval_cleaned.items():

    # Force numeric on all metric columns first
    for col in ["call_volume", "abandoned_calls", "abandoned_rate", "cct", "service_level"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # is_weekend derived from date
    df["is_weekend"] = df["date"].dt.dayofweek >= 5

    for col in ["call_volume", "abandoned_calls", "abandoned_rate", "cct", "service_level"]:
        if col not in df.columns:
            continue

        # Pass 1: (slot_index, is_weekend) group median
        group_med = (
            df.groupby(["slot_index", "is_weekend"])[col]
            .transform("median")
        )
        mask = df[col].isna()
        df.loc[mask, col] = group_med[mask]

        # Pass 2: slot_index-only median fallback
        group_med2 = (
            df.groupby("slot_index")[col]
            .transform("median")
        )
        mask = df[col].isna()
        df.loc[mask, col] = group_med2[mask]

        # Pass 3: global median safety net
        mask = df[col].isna()
        df.loc[mask, col] = df[col].median()

    interval_cleaned[file_name] = df

    remaining = df[["call_volume","cct","abandoned_rate"]].isna().sum().sum()
    print(f"{file_name} | remaining NaN: {remaining}")

A_Interval.csv | remaining NaN: 4368
B_Interval.csv | remaining NaN: 4368
C_Interval.csv | remaining NaN: 4368
D_Interval.csv | remaining NaN: 4368


In [162]:
df = interval_cleaned["A_Interval.csv"]
print("Dtypes:")
print(df[["slot_index","call_volume","cct","abandoned_rate"]].dtypes)
print("\nNon-null counts:")
print(df[["call_volume","cct","abandoned_rate"]].notna().sum())
print("\nSample rows:")
print(df[["date","slot_index","call_volume","cct"]].head(10))

Dtypes:
slot_index          int64
call_volume       float64
cct               float64
abandoned_rate    float64
dtype: object

Non-null counts:
call_volume       4368
cct               4368
abandoned_rate       0
dtype: int64

Sample rows:
        date  slot_index  call_volume     cct
0 2025-04-01           0        127.0  316.24
1 2025-04-01           1        127.0  316.24
2 2025-04-01           2        127.0  316.24
3 2025-04-01           3        127.0  316.24
4 2025-04-01           4        127.0  316.24
5 2025-04-01           5        127.0  316.24
6 2025-04-01           6        127.0  316.24
7 2025-04-01           7        127.0  316.24
8 2025-04-01           8        127.0  316.24
9 2025-04-01           9        127.0  316.24


In [163]:
def normalize_month(s):
    s = s.astype(str).str.strip().str.lower()
    return s.map({
        "april":4, "apr":4, "4":4, "4.0":4,
        "may":5,             "5":5, "5.0":5,
        "june":6, "jun":6,  "6":6, "6.0":6
    })

interval_cleaned = {}
interval_day_diagnostics = []

interval_raw_names = {
    "A_Interval.csv": "A_Interval.csv",
    "B_Interval.csv": "B_Interval.csv",
    "C_Interval.csv": "C_Interval.csv",
    "D_Interval.csv": "D_Interval.csv",
}

for file_name, raw_name in interval_raw_names.items():
    portfolio = get_portfolio(file_name)

    df = pd.read_csv(raw_dir / raw_name)
    df = standardize_columns(df)

    # Fix month
    df["month"] = df["month"].astype(str).str.strip()
    df.loc[df["month"].isin(["", "nan", "None", "NaN"]), "month"] = np.nan
    df["month"] = df["month"].ffill()
    df["month_num"] = normalize_month(df["month"])

    # Fix day
    df["day"] = pd.to_numeric(df["day"], errors="coerce")

    # Parse interval HH:MM:SS → slot_index 0-47
    iv = df["interval"].astype(str).str.strip()
    hour   = pd.to_numeric(iv.str[:2],  errors="coerce")
    minute = pd.to_numeric(iv.str[3:5], errors="coerce")
    # Force to regular int64, not nullable Int64
    df["slot_index"] = (hour * 2 + (minute // 30)).astype("float").astype("Int64")

    # Force all metric columns to numeric now
    for col in ["service_level","call_volume","abandoned_calls","abandoned_rate","cct"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Build date
    df["date"] = pd.to_datetime(
        pd.DataFrame({"year": 2025, "month": df["month_num"], "day": df["day"]}),
        errors="coerce"
    )

    print(f"\n{file_name}")
    print("  Invalid dates:", df["date"].isna().sum())
    print("  Invalid slots:", df["slot_index"].isna().sum())

    # Step 6: build complete 48-slot grid
    base_days = (df[["date"]].dropna().drop_duplicates()
                 .sort_values("date").reset_index(drop=True))
    print("  Unique days:", len(base_days))

    full_grid = base_days.merge(
        pd.DataFrame({"slot_index": np.arange(48, dtype="int64")}),
        how="cross"
    )

    # Force both sides to same dtype before merge
    df["slot_index"]          = df["slot_index"].astype("int64", errors="ignore")
    full_grid["slot_index"]   = full_grid["slot_index"].astype("int64")

    print("  Grid size:", len(full_grid))

    df_valid = df.dropna(subset=["date","slot_index"]).copy()
    df_valid["slot_index"] = df_valid["slot_index"].astype("int64")

    merged = full_grid.merge(df_valid, on=["date","slot_index"], how="left")

    # Check merge actually worked
    print("  call_volume non-null after merge:", merged["call_volume"].notna().sum())

    # Restore readable columns
    merged["month"]               = merged["date"].dt.month
    merged["day"]                 = merged["date"].dt.day
    merged["interval_start_hour"] = merged["slot_index"] // 2
    merged["overnight_flag"]      = merged["interval_start_hour"] < 7
    merged["is_weekend"]          = merged["date"].dt.dayofweek >= 5

    metric_cols = [c for c in ["service_level","call_volume",
                                "abandoned_calls","abandoned_rate","cct"]
                   if c in merged.columns]
    merged["slot_missing_flag"] = merged[metric_cols].isna().all(axis=1)

    if "call_volume" in merged.columns:
        merged["low_volume_flag"] = merged["call_volume"].fillna(0) <= 2

    interval_cleaned[file_name] = merged
    interval_day_diagnostics.append({
        "filename":             file_name,
        "portfolio":            portfolio,
        "rows_after_full_grid": len(merged),
        "unique_days":          merged["date"].nunique(),
        "slot_missing_rows":    int(merged["slot_missing_flag"].sum()),
    })
    print(f"  ✓ Done | shape = {merged.shape}")

interval_day_diagnostics_df = pd.DataFrame(interval_day_diagnostics)
display(interval_day_diagnostics_df)



A_Interval.csv
  Invalid dates: 1
  Invalid slots: 1536
  Unique days: 91
  Grid size: 4368
  call_volume non-null after merge: 2476
  ✓ Done | shape = (4368, 16)

B_Interval.csv
  Invalid dates: 0
  Invalid slots: 1750
  Unique days: 91
  Grid size: 4368
  call_volume non-null after merge: 2482
  ✓ Done | shape = (4368, 16)

C_Interval.csv
  Invalid dates: 0
  Invalid slots: 1824
  Unique days: 91
  Grid size: 4368
  call_volume non-null after merge: 2495
  ✓ Done | shape = (4368, 16)

D_Interval.csv
  Invalid dates: 0
  Invalid slots: 1810
  Unique days: 91
  Grid size: 4368
  call_volume non-null after merge: 2484
  ✓ Done | shape = (4368, 16)


,filename,portfolio,rows_after_full_grid,unique_days,slot_missing_rows
0,A_Interval.csv,A,4368,91,1872
1,B_Interval.csv,B,4368,91,1867
2,C_Interval.csv,C,4368,91,1836
3,D_Interval.csv,D,4368,91,1825


In [165]:
for file_name, df in interval_cleaned.items():

    for col in ["call_volume", "abandoned_calls", "abandoned_rate", "cct", "service_level"]:
        if col not in df.columns:
            continue

        # Pass 1: (slot_index, is_weekend) group median
        group_med = (
            df.groupby(["slot_index", "is_weekend"])[col]
            .transform("median")
        )
        mask = df[col].isna()
        df.loc[mask, col] = group_med[mask]

        # Pass 2: slot_index-only median fallback
        group_med2 = (
            df.groupby("slot_index")[col]
            .transform("median")
        )
        mask = df[col].isna()
        df.loc[mask, col] = group_med2[mask]

        # Pass 3: global median safety net
        mask = df[col].isna()
        df.loc[mask, col] = df[col].median()

    interval_cleaned[file_name] = df

    remaining = df[["call_volume","cct","abandoned_rate"]].isna().sum().sum()
    print(f"{file_name} | remaining NaN: {remaining}")

A_Interval.csv | remaining NaN: 4368
B_Interval.csv | remaining NaN: 4368
C_Interval.csv | remaining NaN: 4368
D_Interval.csv | remaining NaN: 4368


In [166]:
df_raw = pd.read_csv(raw_dir / "A_Interval.csv")
df_raw = standardize_columns(df_raw)

print("abandoned_rate sample values:")
print(df_raw["abandoned_rate"].head(20).tolist())
print("\ndtype:", df_raw["abandoned_rate"].dtype)
print("non-null count:", df_raw["abandoned_rate"].notna().sum())
print("unique values sample:", df_raw["abandoned_rate"].dropna().unique()[:10])

abandoned_rate sample values:
['0.00%', '0.00%', '0.00%', '0.00%', '0.00%', '0.00%', '0.00%', '0.00%', '0.00%', '0.00%', '0.00%', '50.00%', '14.29%', '0.00%', '0.00%', '0.00%', '0.00%', '0.58%', '0.00%', '0.00%']

dtype: object
non-null count: 3919
unique values sample: ['0.00%' '50.00%' '14.29%' '0.58%' '0.41%' '0.39%' '1.54%' '0.77%' '2.34%'
 '1.79%']


In [167]:
for file_name, df in interval_cleaned.items():
    print(f"\n{file_name}")
    for col in ["call_volume", "cct", "abandoned_calls", "abandoned_rate"]:
        if col in df.columns:
            print(f"  {col}: {df[col].isna().sum()} NaN")


A_Interval.csv
  call_volume: 0 NaN
  cct: 0 NaN
  abandoned_calls: 0 NaN
  abandoned_rate: 4368 NaN

B_Interval.csv
  call_volume: 0 NaN
  cct: 0 NaN
  abandoned_calls: 0 NaN
  abandoned_rate: 4368 NaN

C_Interval.csv
  call_volume: 0 NaN
  cct: 0 NaN
  abandoned_calls: 0 NaN
  abandoned_rate: 4368 NaN

D_Interval.csv
  call_volume: 0 NaN
  cct: 0 NaN
  abandoned_calls: 0 NaN
  abandoned_rate: 4368 NaN


In [168]:
for file_name, df in interval_cleaned.items():
    # Recalculate abandoned_rate from source columns
    valid = df["call_volume"] > 0
    df.loc[valid, "abandoned_rate"] = (
        df.loc[valid, "abandoned_calls"] / df.loc[valid, "call_volume"]
    ).clip(0, 1)
    df.loc[~valid, "abandoned_rate"] = 0
    
    interval_cleaned[file_name] = df
    print(f"{file_name} | abandoned_rate NaN: {df['abandoned_rate'].isna().sum()}")

A_Interval.csv | abandoned_rate NaN: 0
B_Interval.csv | abandoned_rate NaN: 0
C_Interval.csv | abandoned_rate NaN: 0
D_Interval.csv | abandoned_rate NaN: 0


In [169]:
for file_name, df in daily_cleaned.items():

    df["_dow"]   = df["date"].dt.dayofweek
    df["_month"] = df["date"].dt.month
    train_rows   = df["is_train_period"] == True

    for col in ["call_volume", "cct", "service_level", "abandon_rate"]:
        if col not in df.columns:
            continue

        # Pass 1: (weekday + month) group median
        dow_month_med = df[train_rows].groupby(["_dow","_month"])[col].median()
        for idx in df[train_rows & df[col].isna()].index:
            key = (df.loc[idx,"_dow"], df.loc[idx,"_month"])
            if key in dow_month_med.index and pd.notna(dow_month_med[key]):
                df.loc[idx, col] = dow_month_med[key]

        # Pass 2: weekday-only median fallback
        dow_med = df[train_rows].groupby("_dow")[col].median()
        for idx in df[train_rows & df[col].isna()].index:
            key = df.loc[idx,"_dow"]
            if key in dow_med.index and pd.notna(dow_med[key]):
                df.loc[idx, col] = dow_med[key]

        # Pass 3: global median safety net
        global_med = df.loc[train_rows, col].median()
        mask = train_rows & df[col].isna()
        df.loc[mask, col] = global_med

    df = df.drop(columns=["_dow","_month"])
    daily_cleaned[file_name] = df

    remaining = df.loc[train_rows, ["call_volume","cct","abandon_rate"]].isna().sum().sum()
    print(f"{file_name} | remaining NaN in train set: {remaining}")

A_Daily.csv | remaining NaN in train set: 578
B_Daily.csv | remaining NaN in train set: 578
C_Daily.csv | remaining NaN in train set: 1156
D_Daily.csv | remaining NaN in train set: 1156


In [170]:
for file_name, df in daily_cleaned.items():
    train_rows = df["is_train_period"] == True
    print(f"\n{file_name}")
    for col in ["call_volume", "cct", "service_level", "abandon_rate"]:
        if col in df.columns:
            na = df.loc[train_rows, col].isna().sum()
            total = train_rows.sum()
            print(f"  {col}: {na} NaN out of {total} train rows")


A_Daily.csv
  call_volume: 0 NaN out of 578 train rows
  cct: 0 NaN out of 578 train rows
  service_level: 578 NaN out of 578 train rows
  abandon_rate: 578 NaN out of 578 train rows

B_Daily.csv
  call_volume: 0 NaN out of 578 train rows
  cct: 0 NaN out of 578 train rows
  service_level: 578 NaN out of 578 train rows
  abandon_rate: 578 NaN out of 578 train rows

C_Daily.csv
  call_volume: 578 NaN out of 578 train rows
  cct: 0 NaN out of 578 train rows
  service_level: 578 NaN out of 578 train rows
  abandon_rate: 578 NaN out of 578 train rows

D_Daily.csv
  call_volume: 578 NaN out of 578 train rows
  cct: 0 NaN out of 578 train rows
  service_level: 578 NaN out of 578 train rows
  abandon_rate: 578 NaN out of 578 train rows


In [171]:
daily_cleaned = {}

daily_raw_names = {
    "A_Daily.csv": "A-Daily.csv",
    "B_Daily.csv": "B-Daily.csv",
    "C_Daily.csv": "C-Daily.csv",
    "D_Daily.csv": "D-Daily.csv",
}

for clean_name, raw_name in daily_raw_names.items():
    portfolio = get_portfolio(clean_name)

    # Load from raw
    df = pd.read_csv(raw_dir / raw_name)
    df = standardize_columns(df)

    # Parse date: "01/01/24 Mon" → keep first token only
    df["date"] = pd.to_datetime(
        df["date"].astype(str).str.strip().str.split().str[0],
        format="%m/%d/%y", errors="coerce"
    )

    # Force all metric columns to numeric
    for col in ["call_volume", "cct", "service_level", "abandon_rate"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Sort
    df = df.sort_values("date").reset_index(drop=True)

    # Time split flags
    df["is_train_period"] = df["date"] <= pd.Timestamp("2025-07-31")
    df["is_aug_holdout"]  = (
        (df["date"] >= pd.Timestamp("2025-08-01")) &
        (df["date"] <= pd.Timestamp("2025-08-31"))
    )
    df["is_post_aug"] = df["date"] > pd.Timestamp("2025-08-31")

    daily_cleaned[clean_name] = df

    train_rows = df["is_train_period"] == True
    print(f"\n{clean_name}")
    for col in ["call_volume", "cct", "service_level", "abandon_rate"]:
        if col in df.columns:
            na = df.loc[train_rows, col].isna().sum()
            print(f"  {col}: {na} NaN out of {train_rows.sum()} train rows")


A_Daily.csv
  call_volume: 573 NaN out of 578 train rows
  cct: 24 NaN out of 578 train rows
  service_level: 578 NaN out of 578 train rows
  abandon_rate: 578 NaN out of 578 train rows

B_Daily.csv
  call_volume: 577 NaN out of 578 train rows
  cct: 29 NaN out of 578 train rows
  service_level: 578 NaN out of 578 train rows
  abandon_rate: 578 NaN out of 578 train rows

C_Daily.csv
  call_volume: 578 NaN out of 578 train rows
  cct: 14 NaN out of 578 train rows
  service_level: 578 NaN out of 578 train rows
  abandon_rate: 578 NaN out of 578 train rows

D_Daily.csv
  call_volume: 578 NaN out of 578 train rows
  cct: 22 NaN out of 578 train rows
  service_level: 578 NaN out of 578 train rows
  abandon_rate: 578 NaN out of 578 train rows


In [172]:
df_raw = pd.read_csv(raw_dir / "A-Daily.csv")
print("Raw column names:", df_raw.columns.tolist())
print("\nFirst 3 rows:")
print(df_raw.head(3).to_string())
print("\nDtypes:")
print(df_raw.dtypes)

Raw column names: ['Date', 'Call Volume', 'CCT', 'Service Level', 'Abandon Rate']

First 3 rows:
           Date Call Volume     CCT Service Level Abandon Rate
0  01/01/24 Mon       2,147  302.45        98.55%        0.37%
1  01/02/24 Tue       7,458  349.22        52.13%       11.36%
2  01/03/24 Wed       6,882  331.07        70.46%        4.32%

Dtypes:
Date              object
Call Volume       object
CCT              float64
Service Level     object
Abandon Rate      object
dtype: object


In [173]:
daily_cleaned = {}

daily_raw_names = {
    "A_Daily.csv": "A-Daily.csv",
    "B_Daily.csv": "B-Daily.csv",
    "C_Daily.csv": "C-Daily.csv",
    "D_Daily.csv": "D-Daily.csv",
}

for clean_name, raw_name in daily_raw_names.items():

    df = pd.read_csv(raw_dir / raw_name)
    df = standardize_columns(df)

    # Parse date
    df["date"] = pd.to_datetime(
        df["date"].astype(str).str.strip().str.split().str[0],
        format="%m/%d/%y", errors="coerce"
    )

    # call_volume: remove commas then convert
    df["call_volume"] = (
        df["call_volume"].astype(str).str.replace(",", "", regex=False)
    )
    df["call_volume"] = pd.to_numeric(df["call_volume"], errors="coerce")

    # cct: already float, just force numeric
    df["cct"] = pd.to_numeric(df["cct"], errors="coerce")

    # service_level: strip % and divide by 100
    df["service_level"] = (
        df["service_level"].astype(str)
        .str.replace("%", "", regex=False)
        .str.strip()
    )
    df["service_level"] = pd.to_numeric(df["service_level"], errors="coerce") / 100

    # abandon_rate: strip % and divide by 100
    df["abandon_rate"] = (
        df["abandon_rate"].astype(str)
        .str.replace("%", "", regex=False)
        .str.strip()
    )
    df["abandon_rate"] = pd.to_numeric(df["abandon_rate"], errors="coerce") / 100

    # Sort
    df = df.sort_values("date").reset_index(drop=True)

    # Time split flags
    df["is_train_period"] = df["date"] <= pd.Timestamp("2025-07-31")
    df["is_aug_holdout"]  = (
        (df["date"] >= pd.Timestamp("2025-08-01")) &
        (df["date"] <= pd.Timestamp("2025-08-31"))
    )
    df["is_post_aug"] = df["date"] > pd.Timestamp("2025-08-31")

    daily_cleaned[clean_name] = df

    train_rows = df["is_train_period"] == True
    print(f"\n{clean_name}")
    for col in ["call_volume", "cct", "service_level", "abandon_rate"]:
        na = df.loc[train_rows, col].isna().sum()
        print(f"  {col}: {na} NaN out of {train_rows.sum()} train rows")


A_Daily.csv
  call_volume: 24 NaN out of 578 train rows
  cct: 24 NaN out of 578 train rows
  service_level: 25 NaN out of 578 train rows
  abandon_rate: 19 NaN out of 578 train rows

B_Daily.csv
  call_volume: 20 NaN out of 578 train rows
  cct: 29 NaN out of 578 train rows
  service_level: 41 NaN out of 578 train rows
  abandon_rate: 14 NaN out of 578 train rows

C_Daily.csv
  call_volume: 20 NaN out of 578 train rows
  cct: 14 NaN out of 578 train rows
  service_level: 14 NaN out of 578 train rows
  abandon_rate: 7 NaN out of 578 train rows

D_Daily.csv
  call_volume: 0 NaN out of 578 train rows
  cct: 22 NaN out of 578 train rows
  service_level: 12 NaN out of 578 train rows
  abandon_rate: 0 NaN out of 578 train rows


In [175]:
for file_name, df in daily_cleaned.items():

    df["_dow"]   = df["date"].dt.dayofweek
    df["_month"] = df["date"].dt.month
    train_rows   = df["is_train_period"] == True

    for col in ["call_volume", "cct", "service_level", "abandon_rate"]:
        if col not in df.columns:
            continue

        # Pass 1: (weekday + month) group median
        dow_month_med = df[train_rows].groupby(["_dow","_month"])[col].median()
        for idx in df[train_rows & df[col].isna()].index:
            key = (df.loc[idx,"_dow"], df.loc[idx,"_month"])
            if key in dow_month_med.index and pd.notna(dow_month_med[key]):
                df.loc[idx, col] = dow_month_med[key]

        # Pass 2: weekday-only median fallback
        dow_med = df[train_rows].groupby("_dow")[col].median()
        for idx in df[train_rows & df[col].isna()].index:
            key = df.loc[idx,"_dow"]
            if key in dow_med.index and pd.notna(dow_med[key]):
                df.loc[idx, col] = dow_med[key]

        # Pass 3: global median safety net
        mask = train_rows & df[col].isna()
        df.loc[mask, col] = df.loc[train_rows, col].median()

    df = df.drop(columns=["_dow","_month"])
    daily_cleaned[file_name] = df

    remaining = df.loc[train_rows, ["call_volume","cct","abandon_rate"]].isna().sum().sum()
    print(f"{file_name} | remaining NaN in train set: {remaining}")


A_Daily.csv | remaining NaN in train set: 0
B_Daily.csv | remaining NaN in train set: 0
C_Daily.csv | remaining NaN in train set: 0
D_Daily.csv | remaining NaN in train set: 0


In [176]:
for col in ["staffing_a","staffing_b","staffing_c","staffing_d"]:
    staffing_cleaned[col] = staffing_cleaned[col].ffill().bfill()

print("Staffing remaining NaN:")
print(staffing_cleaned[["staffing_a","staffing_b","staffing_c","staffing_d"]].isna().sum())

Staffing remaining NaN:
staffing_a    0
staffing_b    0
staffing_c    0
staffing_d    0
dtype: int64


In [177]:
for file_name, df in interval_cleaned.items():

    overnight_low_vol = (
        (df["overnight_flag"] == True) &
        (df["call_volume"] <= 2) &
        (df["cct"] > 1800)
    )

    slot_overnight_med = (
        df[df["overnight_flag"] == True]
        .groupby("slot_index")["cct"]
        .median()
    )

    df.loc[overnight_low_vol, "cct"] = (
        df.loc[overnight_low_vol, "slot_index"]
        .map(slot_overnight_med)
    )

    interval_cleaned[file_name] = df
    print(f"{file_name} | overnight extreme CCT slots smoothed: {overnight_low_vol.sum()}")

A_Interval.csv | overnight extreme CCT slots smoothed: 0
B_Interval.csv | overnight extreme CCT slots smoothed: 0
C_Interval.csv | overnight extreme CCT slots smoothed: 0
D_Interval.csv | overnight extreme CCT slots smoothed: 0


In [178]:
for file_name, df in interval_cleaned.items():
    df["call_volume"]     = df["call_volume"].clip(lower=0)
    df["abandoned_calls"] = df["abandoned_calls"].clip(lower=0)
    df["cct"]             = df["cct"].clip(lower=0)
    df["abandoned_rate"]  = df["abandoned_rate"].clip(lower=0, upper=1)
    df["service_level"]   = df["service_level"].clip(lower=0, upper=1)
    interval_cleaned[file_name] = df

for file_name, df in daily_cleaned.items():
    df["call_volume"]  = df["call_volume"].clip(lower=0)
    df["cct"]          = df["cct"].clip(lower=0)
    df["abandon_rate"] = df["abandon_rate"].clip(lower=0, upper=1)
    daily_cleaned[file_name] = df

print("Hard floors applied.")

Hard floors applied.


In [179]:
phase3_dir = base_dir / "Data_Cleaned" / "Phase3_Final"
phase3_dir.mkdir(parents=True, exist_ok=True)

for file_name, df in daily_cleaned.items():
    df.to_csv(phase3_dir / file_name, index=False)
    print(f"Saved {file_name} | shape = {df.shape}")

for file_name, df in interval_cleaned.items():
    df.to_csv(phase3_dir / file_name, index=False)
    print(f"Saved {file_name} | shape = {df.shape}")

staffing_cleaned.to_csv(phase3_dir / "Daily_Staffing.csv", index=False)
print(f"Saved Daily_Staffing.csv | shape = {staffing_cleaned.shape}")
print("\nAll clean files saved to:", phase3_dir)

Saved A_Daily.csv | shape = (731, 8)
Saved B_Daily.csv | shape = (731, 8)
Saved C_Daily.csv | shape = (731, 8)
Saved D_Daily.csv | shape = (731, 8)
Saved A_Interval.csv | shape = (4368, 16)
Saved B_Interval.csv | shape = (4368, 16)
Saved C_Interval.csv | shape = (4368, 16)
Saved D_Interval.csv | shape = (4368, 16)
Saved Daily_Staffing.csv | shape = (365, 15)

All clean files saved to: /home/sagemaker-user/Data_Cleaned/Phase3_Final


In [182]:
errors   = []
warnings = []

for file_name, df in daily_cleaned.items():
    train_rows = df["is_train_period"] == True

    na = df.loc[train_rows, ["call_volume","cct","abandon_rate"]].isna().sum().sum()
    if na > 0:
        errors.append(f"[{file_name}] {na} NaN in train set")

    for col in ["call_volume","cct"]:
        if (df[col] < 0).sum() > 0:
            errors.append(f"[{file_name}] negatives in {col}")

    leak = df[train_rows & (df["date"].dt.month==8) & (df["date"].dt.year==2025)]
    if len(leak):
        errors.append(f"[{file_name}] LEAKAGE: {len(leak)} Aug 2025 rows in train")

for file_name, df in interval_cleaned.items():
    if len(df) != 4368:
        errors.append(f"[{file_name}] Expected 4368 rows, got {len(df)}")

    for col in ["call_volume","cct","abandoned_rate"]:
        na = df[col].isna().sum()
        if na > 0:
            errors.append(f"[{file_name}] {na} NaN in {col}")

    for col in ["call_volume","abandoned_calls","cct"]:
        if (df[col] < 0).sum() > 0:
            errors.append(f"[{file_name}] negatives in {col}")

    oob = ((df["abandoned_rate"] < 0) | (df["abandoned_rate"] > 1)).sum()
    if oob:
        errors.append(f"[{file_name}] {oob} abandoned_rate out of [0,1]")

    if df["date"].nunique() != 91:
        errors.append(f"[{file_name}] Expected 91 unique days, got {df['date'].nunique()}")

na_staff = staffing_cleaned[["staffing_a","staffing_b","staffing_c","staffing_d"]].isna().sum().sum()
if na_staff > 0:
    warnings.append(f"[Staffing] {na_staff} NaN remaining")

if errors:
    print(f"\n❌ {len(errors)} ERROR(S):")
    for e in errors: print(f"   {e}")
else:
    print("\n✅ NO ERRORS - data is clean and ready for modeling")

if warnings:
    print(f"\n⚠️  {len(warnings)} WARNING(S):")
    for w in warnings: print(f"   {w}")
else:
    print("✅ NO WARNINGS")


✅ NO ERRORS - data is clean and ready for modeling
✅ NO WARNINGS


In [183]:
phase3_dir = base_dir / "Data_Cleaned" / "Phase3_Final"

# ── DAILY MISSING FLAGS ──────────────────────────────────────
daily_raw_names = {
    "A_Daily.csv": "A-Daily.csv",
    "B_Daily.csv": "B-Daily.csv",
    "C_Daily.csv": "C-Daily.csv",
    "D_Daily.csv": "D-Daily.csv",
}

for clean_name, raw_name in daily_raw_names.items():
    # Load raw to see original missingness
    raw = pd.read_csv(raw_dir / raw_name)
    raw = standardize_columns(raw)
    raw["date"] = pd.to_datetime(
        raw["date"].astype(str).str.strip().str.split().str[0],
        format="%m/%d/%y", errors="coerce"
    )

    # Load the already-clean file
    df = daily_cleaned[clean_name].copy()

    # Add one flag per metric column
    for col in ["call_volume", "cct", "service_level", "abandon_rate"]:
        if col in raw.columns:
            raw_col = pd.to_numeric(
                raw[col].astype(str)
                .str.replace(",", "", regex=False)
                .str.replace("%", "", regex=False)
                .str.strip(),
                errors="coerce"
            )
            # Align by date
            raw_missing = raw.set_index("date")[col].isna()
            df[f"{col}_was_missing"] = df["date"].map(raw_missing).fillna(False).astype(int)

    daily_cleaned[clean_name] = df
    df.to_csv(phase3_dir / clean_name, index=False)
    flags = [c for c in df.columns if c.endswith("_was_missing")]
    total_flagged = df[flags].sum().sum()
    print(f"{clean_name} | missing flags added: {flags} | total flagged cells: {total_flagged}")

# ── INTERVAL MISSING FLAGS ────────────────────────────────────
interval_raw_names = {
    "A_Interval.csv": "A_Interval.csv",
    "B_Interval.csv": "B_Interval.csv",
    "C_Interval.csv": "C_Interval.csv",
    "D_Interval.csv": "D_Interval.csv",
}

def normalize_month(s):
    s = s.astype(str).str.strip().str.lower()
    return s.map({
        "april":4,"apr":4,"4":4,"4.0":4,
        "may":5,"5":5,"5.0":5,
        "june":6,"jun":6,"6":6,"6.0":6
    })

for clean_name, raw_name in interval_raw_names.items():
    # Reload raw with correct parsing
    raw = pd.read_csv(raw_dir / raw_name)
    raw = standardize_columns(raw)

    raw["month"] = raw["month"].astype(str).str.strip()
    raw.loc[raw["month"].isin(["","nan","None","NaN"]), "month"] = np.nan
    raw["month"] = raw["month"].ffill()
    raw["month_num"] = normalize_month(raw["month"])
    raw["day"] = pd.to_numeric(raw["day"], errors="coerce")

    iv = raw["interval"].astype(str).str.strip()
    hour   = pd.to_numeric(iv.str[:2], errors="coerce")
    minute = pd.to_numeric(iv.str[3:5], errors="coerce")
    raw["slot_index"] = (hour * 2 + (minute // 30)).astype("float").astype("Int64")

    raw["date"] = pd.to_datetime(
        pd.DataFrame({"year":2025,"month":raw["month_num"],"day":raw["day"]}),
        errors="coerce"
    )

    for col in ["call_volume","abandoned_calls","cct","service_level"]:
        if col in raw.columns:
            raw[col] = pd.to_numeric(raw[col], errors="coerce")

    # Build lookup: (date, slot_index) → was originally missing
    raw["slot_index"] = raw["slot_index"].astype("Int64")
    raw_valid = raw.dropna(subset=["date","slot_index"]).copy()
    raw_valid["slot_index"] = raw_valid["slot_index"].astype(int)

    # Load clean file
    df = interval_cleaned[clean_name].copy()

    for col in ["call_volume","abandoned_calls","cct","service_level"]:
        if col in raw_valid.columns:
            # Mark which (date, slot_index) had missing in raw
            missing_mask = raw_valid.set_index(["date","slot_index"])[col].isna()
            df_idx = df.set_index(["date","slot_index"])
            df[f"{col}_was_missing"] = (
                df.set_index(["date","slot_index"])
                .index.map(lambda x: missing_mask.get(x, True))  # True = was missing (no raw row)
                .astype(int)
            )

    # abandoned_rate was fully recalculated so always flag as derived
    df["abandoned_rate_was_missing"] = 1

    interval_cleaned[clean_name] = df
    df.to_csv(phase3_dir / clean_name, index=False)
    flags = [c for c in df.columns if c.endswith("_was_missing")]
    total_flagged = df[flags].sum().sum()
    print(f"{clean_name} | missing flags added | total flagged cells: {total_flagged}")

print("\n✅ Missing indicator flags added to all files.")

A_Daily.csv | missing flags added: ['call_volume_was_missing', 'cct_was_missing', 'service_level_was_missing', 'abandon_rate_was_missing'] | total flagged cells: 103
B_Daily.csv | missing flags added: ['call_volume_was_missing', 'cct_was_missing', 'service_level_was_missing', 'abandon_rate_was_missing'] | total flagged cells: 126
C_Daily.csv | missing flags added: ['call_volume_was_missing', 'cct_was_missing', 'service_level_was_missing', 'abandon_rate_was_missing'] | total flagged cells: 61
D_Daily.csv | missing flags added: ['call_volume_was_missing', 'cct_was_missing', 'service_level_was_missing', 'abandon_rate_was_missing'] | total flagged cells: 64
A_Interval.csv | missing flags added | total flagged cells: 14435
B_Interval.csv | missing flags added | total flagged cells: 14373
C_Interval.csv | missing flags added | total flagged cells: 14326
D_Interval.csv | missing flags added | total flagged cells: 14352

✅ Missing indicator flags added to all files.


In [187]:
import json
from datetime import datetime

# Readable summary
print("=" * 60)
print("CLEANING LOG SUMMARY")
print("=" * 60)
print(f"Project:    {cleaning_log['project']}")
print(f"Generated:  {cleaning_log['generated']}")
print(f"Train cutoff: {cleaning_log['train_cutoff']}")
print(f"\nIssues found and fixed: {len(cleaning_log['issues_found_and_fixed'])}")
for item in cleaning_log["issues_found_and_fixed"]:
    print(f"  • {item['issue']} - {item['affected']}")
print(f"\nValidation: {cleaning_log['validation_result']['status']}")
print(f"\nLog saved to: {log_path}")

CLEANING LOG SUMMARY
Project:    Synchrony Datathon — Contact Center Forecasting
Generated:  2026-03-28 19:06
Train cutoff: 2025-07-31

Issues found and fixed: 11
  • Date column format - All 4 daily files
  • Call volume stored as string with commas - All 4 daily files
  • Service level and abandon rate stored as percentage strings - All 4 daily files
  • Phase 2 standardization destroyed month and interval columns - All 4 interval files
  • Excel merged cells in month column - All 4 interval files
  • Interval column format HH:MM:SS not numeric - All 4 interval files
  • Incomplete interval grids - A_Interval (worst), B, D partially
  • Abandoned rate stored as percentage string - All 4 interval files
  • Overnight CCT extreme values - All 4 interval files (overnight hours)
  • Staffing column names - Daily_Staffing.csv
  • August 2025 leakage risk - All 4 daily files

Validation: PASSED — data is clean and ready for modeling

Log saved to: /home/sagemaker-user/Data_Cleaned/Phase3_Fi

In [188]:
daily_validation_rows = []

for file_name, df in daily_cleaned.items():
    temp = df.copy()
    portfolio = get_portfolio(file_name)

    date_min = temp["date"].min()
    date_max = temp["date"].max()

    expected_dates = pd.date_range(date_min, date_max, freq="D")
    actual_dates = pd.Index(temp["date"].dropna().sort_values().unique())

    missing_dates = len(expected_dates.difference(actual_dates))
    duplicate_dates = int(temp["date"].duplicated().sum())

    daily_validation_rows.append({
        "filename": file_name,
        "portfolio": portfolio,
        "rows": len(temp),
        "date_min": date_min,
        "date_max": date_max,
        "missing_dates_in_sequence": missing_dates,
        "duplicate_dates": duplicate_dates,
        "negative_call_volume_rows": int(((temp["call_volume"] < 0) & temp["call_volume"].notna()).sum()) if "call_volume" in temp.columns else 0,
        "negative_cct_rows": int(((temp["cct"] < 0) & temp["cct"].notna()).sum()) if "cct" in temp.columns else 0,
        "invalid_abandon_rate_rows": int((((temp["abandon_rate"] < 0) | (temp["abandon_rate"] > 100)) & temp["abandon_rate"].notna()).sum()) if "abandon_rate" in temp.columns else 0,
        "invalid_service_level_rows": int((((temp["service_level"] < 0) | (temp["service_level"] > 100)) & temp["service_level"].notna()).sum()) if "service_level" in temp.columns else 0,
        "train_rows": int(temp["is_train_period"].sum()),
        "aug_holdout_rows": int(temp["is_aug_holdout"].sum()),
        "post_aug_rows": int(temp["is_post_aug"].sum())
    })

daily_validation_df = pd.DataFrame(daily_validation_rows)
display(daily_validation_df)

,filename,portfolio,rows,date_min,date_max,missing_dates_in_sequence,duplicate_dates,negative_call_volume_rows,negative_cct_rows,invalid_abandon_rate_rows,invalid_service_level_rows,train_rows,aug_holdout_rows,post_aug_rows
0,A_Daily.csv,A,731,2024-01-01,2025-12-31,0,0,0,0,0,0,578,31,122
1,B_Daily.csv,B,731,2024-01-01,2025-12-31,0,0,0,0,0,0,578,31,122
2,C_Daily.csv,C,731,2024-01-01,2025-12-31,0,0,0,0,0,0,578,31,122
3,D_Daily.csv,D,731,2024-01-01,2025-12-31,0,0,0,0,0,0,578,31,122


In [189]:
leakage_check_rows = []

for file_name, df in daily_cleaned.items():
    temp = df.copy()
    portfolio = get_portfolio(file_name)

    train_aug_overlap = int(
        ((temp["is_train_period"]) & (temp["date"].between("2025-08-01", "2025-08-31"))).sum()
    )

    leakage_check_rows.append({
        "filename": file_name,
        "portfolio": portfolio,
        "aug_rows_inside_train": train_aug_overlap
    })

leakage_check_df = pd.DataFrame(leakage_check_rows)
display(leakage_check_df)

,filename,portfolio,aug_rows_inside_train
0,A_Daily.csv,A,0
1,B_Daily.csv,B,0
2,C_Daily.csv,C,0
3,D_Daily.csv,D,0


In [190]:
staffing_validation_df = pd.DataFrame([{
    "rows": len(staffing_cleaned),
    "date_min": staffing_cleaned["date"].min(),
    "date_max": staffing_cleaned["date"].max(),
    "invalid_dates": int(staffing_cleaned["date"].isna().sum()),
    "duplicate_dates": int(staffing_cleaned["date"].duplicated().sum()),
    "negative_staffing_a": int(((staffing_cleaned["staffing_a"] < 0) & staffing_cleaned["staffing_a"].notna()).sum()),
    "negative_staffing_b": int(((staffing_cleaned["staffing_b"] < 0) & staffing_cleaned["staffing_b"].notna()).sum()),
    "negative_staffing_c": int(((staffing_cleaned["staffing_c"] < 0) & staffing_cleaned["staffing_c"].notna()).sum()),
    "negative_staffing_d": int(((staffing_cleaned["staffing_d"] < 0) & staffing_cleaned["staffing_d"].notna()).sum())
}])

display(staffing_validation_df)

,rows,date_min,date_max,invalid_dates,duplicate_dates,negative_staffing_a,negative_staffing_b,negative_staffing_c,negative_staffing_d
0,365,NaT,NaT,365,364,0,0,0,0


In [191]:
interval_validation_rows = []

for file_name, df in interval_flagged.items():
    temp = df.copy()
    portfolio = get_portfolio(file_name)

    unique_days = temp["date"].nunique() if "date" in temp.columns else 0
    expected_rows = unique_days * 48

    duplicate_keys = 0
    if {"date", "interval"}.issubset(temp.columns):
        duplicate_keys = int(temp.duplicated(subset=["date", "interval"]).sum())

    interval_validation_rows.append({
        "filename": file_name,
        "portfolio": portfolio,
        "rows": len(temp),
        "unique_days": unique_days,
        "expected_rows_from_days": expected_rows,
        "rows_match_expected": len(temp) == expected_rows,
        "duplicate_date_interval_rows": duplicate_keys,
        "incomplete_days": int(
            temp[["date", "incomplete_day_flag"]].drop_duplicates()["incomplete_day_flag"].sum()
        ) if {"date", "incomplete_day_flag"}.issubset(temp.columns) else np.nan,
        "slot_missing_rows": int(temp["slot_missing_flag"].sum()) if "slot_missing_flag" in temp.columns else np.nan
    })

interval_validation_df = pd.DataFrame(interval_validation_rows)
display(interval_validation_df)

,filename,portfolio,rows,unique_days,expected_rows_from_days,rows_match_expected,duplicate_date_interval_rows,incomplete_days,slot_missing_rows
0,A_Interval.csv,A,1488,0,0,False,0,NaN,1488
1,B_Interval.csv,B,1488,0,0,False,0,NaN,1488
2,C_Interval.csv,C,1488,0,0,False,0,NaN,1488
3,D_Interval.csv,D,1488,0,0,False,0,NaN,1488


In [192]:
consistency_rows = []

for file_name, df in interval_flagged.items():
    temp = df.copy()
    portfolio = get_portfolio(file_name)

    if {"abandoned_calls", "abandoned_rate", "call_volume"}.issubset(temp.columns):
        expected_abandoned = temp["call_volume"] * (temp["abandoned_rate"] / 100.0)
        gap = (temp["abandoned_calls"] - expected_abandoned).abs()

        consistency_rows.append({
            "filename": file_name,
            "portfolio": portfolio,
            "non_negative_call_volume": int(((temp["call_volume"] >= 0) | temp["call_volume"].isna()).all()),
            "non_negative_abandoned_calls": int(((temp["abandoned_calls"] >= 0) | temp["abandoned_calls"].isna()).all()),
            "non_negative_cct": int(((temp["cct"] >= 0) | temp["cct"].isna()).all()) if "cct" in temp.columns else 1,
            "abandoned_rate_in_bounds": int((((temp["abandoned_rate"].between(0, 100)) | temp["abandoned_rate"].isna())).all()),
            "mismatch_rows_gt_1": int((gap > 1.0).sum()),
            "mean_gap": float(gap.mean(skipna=True))
        })

consistency_df = pd.DataFrame(consistency_rows)
display(consistency_df)

,filename,portfolio,non_negative_call_volume,non_negative_abandoned_calls,non_negative_cct,abandoned_rate_in_bounds,mismatch_rows_gt_1,mean_gap
0,A_Interval.csv,A,1,1,1,1,0,NaN
1,B_Interval.csv,B,1,1,1,1,0,NaN
2,C_Interval.csv,C,1,1,1,1,0,NaN
3,D_Interval.csv,D,1,1,1,1,0,NaN


In [193]:
final_quality_gate = pd.DataFrame({
    "check": [
        "Daily dates continuous",
        "No daily duplicates",
        "No August rows inside train",
        "Staffing dates valid",
        "Interval rows match days*48",
        "No duplicate (date, interval) rows",
        "Rates in valid bounds",
        "Counts non-negative",
        "Abandoned metrics internally consistent"
    ],
    "status": [
        "PASS" if (daily_validation_df["missing_dates_in_sequence"] == 0).all() else "FAIL",
        "PASS" if (daily_validation_df["duplicate_dates"] == 0).all() else "FAIL",
        "PASS" if (leakage_check_df["aug_rows_inside_train"] == 0).all() else "FAIL",
        "PASS" if int(staffing_validation_df["invalid_dates"].iloc[0]) == 0 and int(staffing_validation_df["duplicate_dates"].iloc[0]) == 0 else "FAIL",
        "PASS" if interval_validation_df["rows_match_expected"].all() else "FAIL",
        "PASS" if (interval_validation_df["duplicate_date_interval_rows"] == 0).all() else "FAIL",
        "PASS" if consistency_df["abandoned_rate_in_bounds"].all() else "FAIL",
        "PASS" if consistency_df["non_negative_call_volume"].all() and consistency_df["non_negative_abandoned_calls"].all() and consistency_df["non_negative_cct"].all() else "FAIL",
        "PASS" if (consistency_df["mismatch_rows_gt_1"] == 0).all() else "FAIL"
    ]
})

display(final_quality_gate)

,check,status
0,Daily dates continuous,PASS
1,No daily duplicates,PASS
2,No August rows inside train,PASS
3,Staffing dates valid,FAIL
4,Interval rows match days*48,FAIL
5,"No duplicate (date, interval) rows",PASS
6,Rates in valid bounds,PASS
7,Counts non-negative,PASS
8,Abandoned metrics internally consistent,PASS


In [195]:
final_dir = base_dir / "Data_Cleaned" / "Phase3_Final"

print("=" * 60)
print("READING FROM PHASE3_FINAL - THE ACTUAL SAVED FILES")
print("=" * 60)

# Check interval files
for p in ["A","B","C","D"]:
    df = pd.read_csv(final_dir / f"{p}_Interval.csv", parse_dates=["date"])
    print(f"\n{p}_Interval.csv")
    print(f"  rows: {len(df)} | unique days: {df['date'].nunique()}")
    print(f"  date min: {df['date'].min()} | date max: {df['date'].max()}")

# Check staffing raw format
print("\n" + "=" * 60)
staff_raw = pd.read_csv(raw_dir / "Daily_Staffing.csv")
print("Staffing raw columns:", staff_raw.columns.tolist())
print("Staffing first column sample:", staff_raw.iloc[:5, 0].tolist())

# Check staffing saved file
staff_saved = pd.read_csv(final_dir / "Daily_Staffing.csv")
print("\nStaffing saved columns:", staff_saved.columns.tolist())
print("Staffing saved date sample:", staff_saved["date"].head(5).tolist())

READING FROM PHASE3_FINAL - THE ACTUAL SAVED FILES

A_Interval.csv
  rows: 4368 | unique days: 91
  date min: 2025-04-01 00:00:00 | date max: 2025-06-30 00:00:00

B_Interval.csv
  rows: 4368 | unique days: 91
  date min: 2025-04-01 00:00:00 | date max: 2025-06-30 00:00:00

C_Interval.csv
  rows: 4368 | unique days: 91
  date min: 2025-04-01 00:00:00 | date max: 2025-06-30 00:00:00

D_Interval.csv
  rows: 4368 | unique days: 91
  date min: 2025-04-01 00:00:00 | date max: 2025-06-30 00:00:00

Staffing raw columns: ['Unnamed: 0', 'A', 'B', 'C', 'D']
Staffing first column sample: ['1/1/25', '1/2/25', '1/3/25', '1/4/25', '1/5/25']

Staffing saved columns: ['date', 'staffing_a', 'staffing_b', 'staffing_c', 'staffing_d', 'duplicate_date_flag', 'invalid_date_flag', 'staffing_a_missing_flag', 'staffing_a_negative_flag', 'staffing_b_missing_flag', 'staffing_b_negative_flag', 'staffing_c_missing_flag', 'staffing_c_negative_flag', 'staffing_d_missing_flag', 'staffing_d_negative_flag']
Staffing sav

In [196]:
# Staffing from raw with correct format
staff_raw = pd.read_csv(raw_dir / "Daily_Staffing.csv")

staffing_cleaned = staff_raw.rename(columns={
    "Unnamed: 0": "date",
    "A": "staffing_a",
    "B": "staffing_b",
    "C": "staffing_c",
    "D": "staffing_d"
})

# Parse with explicit format — '1/1/25' is %m/%d/%y
staffing_cleaned["date"] = pd.to_datetime(
    staffing_cleaned["date"], format="%m/%d/%y", errors="coerce"
)

staffing_cleaned = staffing_cleaned.sort_values("date").reset_index(drop=True)

# Numeric conversion
for col in ["staffing_a","staffing_b","staffing_c","staffing_d"]:
    staffing_cleaned[col] = pd.to_numeric(staffing_cleaned[col], errors="coerce")

# Fill any gaps
for col in ["staffing_a","staffing_b","staffing_c","staffing_d"]:
    staffing_cleaned[col] = staffing_cleaned[col].ffill().bfill()

# Verifing
print("Date min:", staffing_cleaned["date"].min())
print("Date max:", staffing_cleaned["date"].max())
print("Invalid dates:", staffing_cleaned["date"].isna().sum())
print("Shape:", staffing_cleaned.shape)

staffing_cleaned.to_csv(final_dir / "Daily_Staffing.csv", index=False)
print("Staffing saved.")


Date min: 2025-01-01 00:00:00
Date max: 2025-12-31 00:00:00
Invalid dates: 0
Shape: (365, 5)
Staffing saved.


In [197]:
staff_raw = pd.read_csv(raw_dir / "Daily_Staffing.csv")
display(staff_raw.head(10))
print(staff_raw.columns.tolist())

,Unnamed: 0,A,B,C,D
0,1/1/25,47.0,75.0,353.0,143.0
1,1/2/25,82.0,184.0,491.0,195.0
2,1/3/25,92.0,186.0,462.0,183.0
3,1/4/25,70.0,148.0,352.0,155.0
4,1/5/25,40.0,110.0,224.0,98.0
5,1/6/25,101.0,190.0,498.0,196.0
6,1/7/25,NaN,NaN,NaN,188.0
7,1/8/25,NaN,NaN,NaN,168.0
8,1/9/25,NaN,NaN,NaN,177.0
9,1/10/25,92.0,166.0,409.0,179.0


['Unnamed: 0', 'A', 'B', 'C', 'D']


In [198]:
raw_a = pd.read_csv(raw_dir / "A_Interval.csv")
display(raw_a.head(20))
print(raw_a.columns.tolist())

,Month,Day,Interval,Service Level,Call Volume,Abandoned Calls,Abandoned Rate,CCT
0,April,1.0,0:00,100.00%,5.0,0.0,0.00%,137.6
1,April,1.0,0:30,100.00%,5.0,0.0,0.00%,263.4
2,April,1.0,1:00,100.00%,4.0,0.0,0.00%,333.25
3,April,1.0,1:30,100.00%,3.0,0.0,0.00%,170
4,April,1.0,2:00,100.00%,1.0,0.0,0.00%,667
5,April,1.0,2:30,100.00%,4.0,0.0,0.00%,416.5
6,April,1.0,3:00,100.00%,1.0,0.0,0.00%,128
7,April,1.0,3:30,100.00%,4.0,0.0,0.00%,158
8,April,1.0,5:00,NaN,NaN,NaN,0.00%,59
9,April,1.0,5:30,NaN,NaN,NaN,0.00%,178


['Month', 'Day', 'Interval', 'Service Level', 'Call Volume', 'Abandoned Calls', 'Abandoned Rate', 'CCT']


In [200]:
final_dir = base_dir / "Data_Cleaned" / "Phase3_Final"

daily    = {}
interval = {}

for p in ["A","B","C","D"]:
    interval[p] = pd.read_csv(final_dir / f"{p}_Interval.csv", parse_dates=["date"])
    daily[p]    = pd.read_csv(final_dir / f"{p}_Daily.csv",    parse_dates=["date"])

staffing = pd.read_csv(final_dir / "Daily_Staffing.csv", parse_dates=["date"])

print("Reloaded all files from Phase3_Final.")
print("Staffing date min:", staffing["date"].min())
print("Staffing date max:", staffing["date"].max())
print("Staffing invalid dates:", staffing["date"].isna().sum())

for p in ["A","B","C","D"]:
    print(f"\nDaily {p}: {daily[p].shape} | date range: {daily[p]['date'].min().date()} to {daily[p]['date'].max().date()}")
    print(f"Interval {p}: {interval[p].shape} | unique days: {interval[p]['date'].nunique()}")

Reloaded all files from Phase3_Final.
Staffing date min: 2025-01-01 00:00:00
Staffing date max: 2025-12-31 00:00:00
Staffing invalid dates: 0

Daily A: (731, 12) | date range: 2024-01-01 to 2025-12-31
Interval A: (4368, 21) | unique days: 91

Daily B: (731, 12) | date range: 2024-01-01 to 2025-12-31
Interval B: (4368, 21) | unique days: 91

Daily C: (731, 12) | date range: 2024-01-01 to 2025-12-31
Interval C: (4368, 21) | unique days: 91

Daily D: (731, 12) | date range: 2024-01-01 to 2025-12-31
Interval D: (4368, 21) | unique days: 91


In [202]:
checks = []

# Check 0: Daily dates continuous
for p in ["A","B","C","D"]:
    df = daily[p].copy()
    df = df.sort_values("date")
    expected = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
    actual   = pd.DatetimeIndex(df["date"])
    missing  = expected.difference(actual)
    checks.append({
        "check": "Daily dates continuous",
        "portfolio": p,
        "status": "PASS" if len(missing) == 0 else f"FAIL — {len(missing)} missing dates"
    })

# Check 1: No daily duplicates
for p in ["A","B","C","D"]:
    dups = daily[p]["date"].duplicated().sum()
    checks.append({
        "check": "No daily duplicates",
        "portfolio": p,
        "status": "PASS" if dups == 0 else f"FAIL — {dups} duplicates"
    })

# Check 2: No August rows inside train
for p in ["A","B","C","D"]:
    df = daily[p]
    train = df[df["is_train_period"] == True]
    aug_in_train = train[
        (train["date"].dt.month == 8) &
        (train["date"].dt.year == 2025)
    ]
    checks.append({
        "check": "No August rows inside train",
        "portfolio": p,
        "status": "PASS" if len(aug_in_train) == 0 else f"FAIL — {len(aug_in_train)} rows"
    })

# Check 3: Staffing dates valid
invalid = staffing["date"].isna().sum()
dups    = staffing["date"].duplicated().sum()
checks.append({
    "check": "Staffing dates valid",
    "portfolio": "ALL",
    "status": "PASS" if invalid == 0 and dups == 0 else f"FAIL — invalid: {invalid}, dups: {dups}"
})

# Check 4: Interval rows match days*48
for p in ["A","B","C","D"]:
    df        = interval[p]
    unique_days = df["date"].nunique()
    expected  = unique_days * 48
    actual    = len(df)
    checks.append({
        "check": "Interval rows match days*48",
        "portfolio": p,
        "status": "PASS" if actual == expected else f"FAIL — got {actual}, expected {expected}"
    })

# Check 5: No duplicate (date, slot_index) rows
for p in ["A","B","C","D"]:
    dups = interval[p].duplicated(subset=["date","slot_index"]).sum()
    checks.append({
        "check": "No duplicate (date, slot_index) rows",
        "portfolio": p,
        "status": "PASS" if dups == 0 else f"FAIL — {dups} duplicates"
    })

# Check 6: Rates in valid bounds
for p in ["A","B","C","D"]:
    df  = interval[p]
    oob = ((df["abandoned_rate"] < 0) | (df["abandoned_rate"] > 1)).sum()
    checks.append({
        "check": "Rates in valid bounds",
        "portfolio": p,
        "status": "PASS" if oob == 0 else f"FAIL — {oob} out of bounds"
    })

# Check 7: Counts non-negative
for p in ["A","B","C","D"]:
    df  = interval[p]
    neg = ((df["call_volume"] < 0) | (df["cct"] < 0) | (df["abandoned_calls"] < 0)).sum()
    checks.append({
        "check": "Counts non-negative",
        "portfolio": p,
        "status": "PASS" if neg == 0 else f"FAIL — {neg} negative values"
    })

# Check 8: Abandoned metrics internally consistent
for p in ["A","B","C","D"]:
    df    = interval[p]
    valid = df["call_volume"] > 0
    computed  = (df.loc[valid,"abandoned_calls"] / df.loc[valid,"call_volume"]).round(4)
    stored    = df.loc[valid,"abandoned_rate"].round(4)
    mismatch  = (computed - stored).abs() > 0.01
    checks.append({
        "check": "Abandoned metrics internally consistent",
        "portfolio": p,
        "status": "PASS" if mismatch.sum() == 0 else f"FAIL — {mismatch.sum()} mismatches"
    })

# Print results
results_df = pd.DataFrame(checks)
display(results_df)

fails = results_df[results_df["status"] != "PASS"]
if len(fails) == 0:
    print("\n✅ ALL CHECKS PASSED - data is clean and ready for modeling")
else:
    print(f"\n❌ {len(fails)} check(s) failed:")
    display(fails)

,check,portfolio,status
0,Daily dates continuous,A,PASS
1,Daily dates continuous,B,PASS
2,Daily dates continuous,C,PASS
3,Daily dates continuous,D,PASS
4,No daily duplicates,A,PASS
5,No daily duplicates,B,PASS
6,No daily duplicates,C,PASS
7,No daily duplicates,D,PASS
8,No August rows inside train,A,PASS
9,No August rows inside train,B,PASS



✅ ALL CHECKS PASSED - data is clean and ready for modeling
